# SalmoTrace-BERT — Pipeline Forensik Genomik *Salmonella*

**Berbasis Whole-Genome SNP + DNABERT-2 Embedding**

> Setiap stage pipeline disimpan sebagai **artifact**. Saat run berikutnya, notebook cukup load artifact — tidak perlu compute ulang dari nol.

```
data/interim/       ← metadata bersih (.parquet)
data/processed/     ← windows FASTA (.npy)
artifacts/
  matrices/         ← SNP matrix (.parquet), distance matrix (.csv)
  embeddings/       ← DNABERT embeddings (.npz)
  models/           ← model RF/SVM (.joblib)
  reports/          ← metrics.json, manifest.json
outputs/figures/    ← semua plot (.png)
```

## Anggota

| No | Nama | NIM |
|:--:|------|:---:|
| 1 | Danendra Shafi Athallah | 13523136 |
| 2 | Muhammad Raihaan Perdana | 13523124 |
| 3 | M. Abizzar Gamadrian | 13523155 |
| 4 | Kenzie Raffa Ardhana | 18223127 |

 
Institut Teknologi Bandung - Komputasi Domain Spesifik

---

## Latar Belakang & Target Eksperimen

### Permasalahan Utama

Kehadiran *Salmonella enterica* dalam rantai pangan bukan sekadar pertanyaan *"apakah bakteri ada?"*, melainkan **"dari mana sumber kontaminasinya?"** Identifikasi sumber isolat — apakah berasal dari hewan ternak, produk pangan olahan, lingkungan, atau kasus klinis manusia — merupakan kunci dalam investigasi forensik genomik dan pengendalian wabah.

Studi *Whole-Genome Sequencing* (WGS) *Salmonella* telah membuktikan bahwa variasi genomik antar-isolat menyimpan sinyal yang cukup kuat untuk membedakan sumber asal, dan pendekatan *machine learning* seperti **Random Forest** telah berhasil digunakan untuk klasifikasi sumber (*source attribution*) berdasarkan fitur genomik tersebut.

---

### Target Eksperimen

**Target utama:** memprediksi dan mengelompokkan sumber isolat *Salmonella enterica* berdasarkan pola genomiknya (*source attribution*).

Label sumber diperoleh dari metadata publik **NCBI Pathogen Detection** (kolom `isolation_source`), kemudian disederhanakan menjadi empat kelas:

| Metadata Asli (contoh) | Label yang Digunakan |
|---|:---:|
| chicken, poultry, egg, meat | `food_animal` |
| food, ready-to-eat, vegetable, dairy | `food` |
| environment, water, facility, surface | `environment` |
| human, clinical, patient | `human` |

Target variabel model:

```
y = source_group  →  {food, food_animal, environment, human}
```

> Jika data terlalu sedikit atau tidak seimbang, dapat disederhanakan menjadi klasifikasi biner: `food_related` vs `non_food_related`.

---

### Formulasi Eksperimen

> Target eksperimen penelitian ini adalah melakukan **source attribution** terhadap isolat *Salmonella enterica* berdasarkan fitur genomik. Label sumber isolat diperoleh dari metadata publik, kemudian disederhanakan menjadi beberapa kelas seperti *food*, *food-animal*, *environment*, dan *human*. Whole-genome SNP alignment digunakan untuk menghasilkan fitur SNP dan jarak genomik antar-isolat, sedangkan **DNABERT-2 embedding** digunakan sebagai representasi tambahan dari sekuens DNA. Model dievaluasi untuk melihat apakah fitur SNP, DNABERT, atau kombinasi keduanya mampu membantu klasifikasi sumber isolat serta mendukung interpretasi kedekatan genomik melalui clustering.

---

### Desain Eksperimen

| Eksperimen | Fitur | Model / Metode | Target / Tujuan |
|:---:|---|---|---|
| E1 | SNP distance | Hierarchical clustering | Visualisasi kedekatan genomik |
| E2 | SNP matrix | Random Forest / SVM | `source_group` |
| E3 | DNABERT embedding | Random Forest / SVM | `source_group` |
| E4 | SNP + DNABERT | Random Forest / SVM | `source_group` |
| E5 *(opsional)* | SNP distance | Clustering validation | Bandingkan dengan `snp_cluster` NCBI |

Data yang digunakan: **genome sequence**, **metadata**, **AMR information**, dan **SNP cluster information** dari NCBI Pathogen Detection.

---

##  Setup & Konfigurasi

In [ ]:
# ── Satu flag untuk mengontrol seluruh checkpoint ─────────────────────────
# False → load artifact jika sudah ada  (hemat waktu untuk demo/laporan)
# True  → compute ulang dari awal semua tahap
FORCE_RECOMPUTE = True

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.titlesize': 13})
sns.set_style('whitegrid')
PALETTE = sns.color_palette('tab10')

FIG_OUT = os.path.join(ROOT, 'outputs', 'figures')
for sub in ['clustering', 'embedding', 'classification']:
    os.makedirs(os.path.join(FIG_OUT, sub), exist_ok=True)

from utils.config import load_config
from utils.checkpoint import (
    load_or_compute,
    save_parquet, load_parquet,
    save_csv_ckpt, load_csv_ckpt,
    save_embeddings, load_embeddings,
    save_model, load_model,
    save_json, load_json,
    save_manifest,
)
from utils.seed import set_global_seed

cfg = load_config(os.path.join(ROOT, 'config.yaml'))
ART = cfg['artifacts']
TARGET_COL = cfg['ml']['target_col']

# ── Global reproducibility seed ───────────────────────────────────────────────
# Must be called before any data splitting, model training, UMAP/PCA, or embedding.
# Seed value comes from config so experiments are always config-controlled.
SEED = cfg.get('project', {}).get('random_state', cfg['ml'].get('random_state', 42))
set_global_seed(SEED)

def _p(rel):   # resolve config-relative path to absolute
    return os.path.join(ROOT, rel)

print('Config loaded. Target kolom ML:', TARGET_COL)
print('FORCE_RECOMPUTE =', FORCE_RECOMPUTE)

---
## Stage 1 — Preprocessing Metadata

**Checkpoint:** `data/interim/clean_metadata.parquet`

| Langkah | Fungsi |
|---|---|
| Filter organisme | `filter_organism` |
| Hapus baris tanpa accession | `drop_missing_accession` |
| Normalisasi isolation_source | `normalize_isolation_source` |
| Buang sumber ambigu | `remove_ambiguous_sources` |
| Pilih serovar dominan | `select_dominant_serovars` |
| Batasi jumlah isolat | `filter_metadata` |
| Cek class balance | `check_class_balance` |

In [ ]:
from data import (
    load_metadata,
    filter_organism, drop_missing_accession,
    normalize_isolation_source, remove_ambiguous_sources,
    select_dominant_serovars, filter_metadata, check_class_balance,
    add_source_group, add_binary_source,
)

meta_cfg = cfg['metadata']

def _build_metadata():
    df = load_metadata(_p(cfg['data']['metadata_path']))
    df = drop_missing_accession(df)
    df = filter_organism(df, meta_cfg['organism'])
    df = normalize_isolation_source(df)
    df = remove_ambiguous_sources(df)
    df = select_dominant_serovars(df, top_n=meta_cfg['top_serovars'])
    df = add_source_group(df)
    df = add_binary_source(df)
    df = filter_metadata(
        df,
        min_isolates=cfg['pipeline']['n_isolates_min'],
        max_isolates=cfg['pipeline']['n_isolates_max'],
    )
    return df

metadata_df = load_or_compute(
    path=_p(ART['metadata_clean_path']),
    compute_fn=_build_metadata,
    save_fn=save_parquet,
    load_fn=load_parquet,
    force=FORCE_RECOMPUTE,
)

# Patch: jika parquet cache tidak punya kolom baru, tambahkan dan simpan ulang
_needs_patch = ('source_binary' not in metadata_df.columns or
               'source_group' not in metadata_df.columns)
if _needs_patch:
    if 'source_group' not in metadata_df.columns:
        metadata_df = add_source_group(metadata_df)
    if 'source_binary' not in metadata_df.columns:
        metadata_df = add_binary_source(metadata_df)
    save_parquet(metadata_df, _p(ART['metadata_clean_path']))
    print('[PATCH] source_group/source_binary ditambahkan ke parquet cache dan disimpan ulang.')

check_class_balance(metadata_df, col=TARGET_COL)

# acc_to_cluster: dipakai di semua EDA cell berikutnya
acc_to_cluster = (
    metadata_df.set_index('assembly_accession')['snp_cluster']
    .fillna('unknown').to_dict()
)

print(f'\nMetadata bersih: {metadata_df.shape}')
metadata_df.head(3)


### EDA — Metadata

In [ ]:
GENOMES_DIR   = _p(cfg['data']['raw_genomes_dir'])
available_fna = {f.replace('.fna','') for f in os.listdir(GENOMES_DIR) if f.endswith('.fna')}
metadata_df['fasta_ok'] = metadata_df['assembly_accession'].isin(available_fna)
ref_path = os.path.join(GENOMES_DIR, cfg['data']['reference_genome'] + '.fna')
print(f'FASTA tersedia   : {metadata_df["fasta_ok"].sum()}/{len(metadata_df)} isolat')
print(f'Reference genome : {"OK" if os.path.exists(ref_path) else "NOT FOUND"}')
print(f'SNP cluster unik : {metadata_df["snp_cluster"].nunique()}')
print(f'Serovar unik     : {metadata_df["serovar"].nunique()}')

In [ ]:
print('='*55)
print(f'  Total isolat     : {len(metadata_df)}')
print(f'  SNP cluster unik : {metadata_df["snp_cluster"].nunique()}')
print(f'  Negara unik      : {metadata_df["geo_loc_name"].nunique()}')
cdate_min = metadata_df['collection_date'].min()
cdate_max = metadata_df['collection_date'].max()
print(f'  Rentang tanggal  : {cdate_min} s/d {cdate_max}')
print('='*55)
metadata_df.describe(include='all').T[['count','unique','top','freq']]

In [ ]:
missing = metadata_df.isnull().sum().rename('missing')
pct     = (missing / len(metadata_df) * 100).rename('%')
mv = pd.concat([missing, pct], axis=1)
print(mv)

fig, ax = plt.subplots(figsize=(9, 3.5))
cols_na = mv[mv['missing'] > 0]
if len(cols_na):
    ax.bar(cols_na.index, cols_na['%'], color='tomato', edgecolor='white')
    ax.set_ylabel('Missing (%)'); ax.set_title('Missing Values per Kolom Metadata')
    for i, (col, row) in enumerate(cols_na.iterrows()):
        ax.text(i, row['%']+0.5, f"{row['missing']}\n({row['%']:.0f}%)",
                ha='center', fontsize=9)
else:
    ax.text(0.5, 0.5, 'Tidak ada missing value!', transform=ax.transAxes,
            ha='center', va='center', fontsize=14, color='green')
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(FIG_OUT, '02_missing_values.png'), dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
src_counts = metadata_df['isolation_source'].fillna('unknown').value_counts()
colors = sns.color_palette('Spectral', len(src_counts))
axes[0].barh(src_counts.index, src_counts.values, color=colors, edgecolor='white')
axes[0].set_xlabel('Jumlah Isolat')
axes[0].set_title('Isolation Source (setelah normalisasi)')
for i, v in enumerate(src_counts.values):
    axes[0].text(v+0.1, i, str(v), va='center', fontsize=9)

axes[1].pie(src_counts.values, labels=src_counts.index, autopct='%1.0f%%',
            colors=sns.color_palette('Set2', len(src_counts)),
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Distribusi Isolation Source')
plt.suptitle('Distribusi Sumber Isolasi', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIG_OUT, '02_source_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
snp_counts = metadata_df['snp_cluster'].fillna('unknown').value_counts()
axes[0].bar(range(len(snp_counts)), snp_counts.values,
            color=sns.color_palette('tab20', len(snp_counts)), edgecolor='white')
axes[0].set_xticks(range(len(snp_counts)))
axes[0].set_xticklabels(snp_counts.index, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Jumlah Isolat')
axes[0].set_title(f'SNP Cluster Distribution ({len(snp_counts)} cluster)')
for i, v in enumerate(snp_counts.values):
    axes[0].text(i, v+0.2, str(v), ha='center', fontsize=8)

metadata_df['year'] = pd.to_datetime(metadata_df['collection_date'], errors='coerce').dt.year
year_counts = metadata_df['year'].dropna().astype(int).value_counts().sort_index()
axes[1].bar(year_counts.index, year_counts.values, color=PALETTE[2], edgecolor='white')
axes[1].set_xlabel('Tahun'); axes[1].set_ylabel('Jumlah Isolat')
axes[1].set_title('Distribusi Tahun Koleksi')
plt.suptitle('SNP Cluster & Tahun Koleksi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_OUT, '02_snp_cluster_year.png'), dpi=150)
plt.show()

In [ ]:
geo_counts = (metadata_df['geo_loc_name'].fillna('Unknown')
              .str.split(':').str[0].str.strip().value_counts().head(15))
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(geo_counts.index, geo_counts.values,
        color=sns.color_palette('Blues_r', len(geo_counts)), edgecolor='white')
ax.set_xlabel('Jumlah Isolat'); ax.set_title('Top 15 Lokasi Geografis Isolat')
for i, v in enumerate(geo_counts.values):
    ax.text(v+0.1, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIG_OUT, '02_geo_distribution.png'), dpi=150)
plt.show()

---
## Stage 2 — Loading FASTA & Genome QC

**Checkpoint:** `data/interim/sequence_quality.csv`

| Langkah | Fungsi |
|---|---|
| Load FASTA → `genomes`, `contig_counts` | `load_genomes` |
| Laporan QC per isolat | `genome_qc_report` |
| Buang isolat gagal QC | `filter_by_qc` |
| Sinkronisasi metadata | `validate_accessions` |

> `genomes` selalu di-load ulang dari FASTA (cepat, ~30 detik). Hanya `qc_df` yang di-checkpoint.

In [ ]:
from data import load_genomes, genome_qc_report, filter_by_qc, validate_accessions

qc_cfg     = cfg['genome_qc']
accessions = metadata_df['assembly_accession'].tolist()
genomes, contig_counts = load_genomes(GENOMES_DIR, accessions)

qc_df = load_or_compute(
    path=_p(ART['sequence_quality_path']),
    compute_fn=lambda: genome_qc_report(
        genomes, contig_counts,
        min_bp      = qc_cfg['min_genome_bp'],
        max_bp      = qc_cfg['max_genome_bp'],
        max_n_frac  = qc_cfg['max_n_fraction'],
        max_contigs = qc_cfg['max_contig_count'],
    ),
    save_fn=save_csv_ckpt,
    load_fn=load_csv_ckpt,
    force=FORCE_RECOMPUTE,
)

genomes     = filter_by_qc(genomes, qc_df)
metadata_df = validate_accessions(metadata_df, genomes)
acc_to_cluster = (
    metadata_df.set_index('assembly_accession')['snp_cluster']
    .fillna('unknown').to_dict()
)
print(f'\nIsolat valid pasca-QC: {len(genomes)}')
qc_df.head(5)

### EDA — Kualitas Sekuens Genome

In [ ]:
# df_seq: gabung qc_df dengan metadata
df_seq = qc_df.reset_index().merge(
    metadata_df[['assembly_accession', 'isolation_source', 'snp_cluster']],
    on='assembly_accession', how='left'
)
if 'assembly_accession' in df_seq.columns and df_seq.index.name != 'assembly_accession':
    pass  # keep as is
df_seq['length_Mb'] = df_seq['genome_length_bp'] / 1e6
print(df_seq[['genome_length_bp','gc_content','n_fraction','contig_count']].describe().round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(df_seq['length_Mb'], bins=15, color=PALETTE[0], edgecolor='white', alpha=0.85)
axes[0].axvline(df_seq['length_Mb'].mean(), color='red', ls='--',
                label=f"Mean: {df_seq['length_Mb'].mean():.3f} Mb")
axes[0].set_xlabel('Panjang Genome (Mb)'); axes[0].set_ylabel('Jumlah Isolat')
axes[0].set_title('Distribusi Panjang Genome'); axes[0].legend()

df_seq.boxplot(column='length_Mb', by='isolation_source', ax=axes[1],
               boxprops=dict(color=PALETTE[0]), medianprops=dict(color='red', linewidth=2),
               whiskerprops=dict(color='gray'),
               flierprops=dict(marker='o', markerfacecolor='orange', markersize=6))
axes[1].set_xlabel('Isolation Source'); axes[1].set_ylabel('Panjang (Mb)')
axes[1].set_title('Panjang Genome per Sumber Isolasi'); plt.suptitle('')

q1, q3 = df_seq['genome_length_bp'].quantile([0.25, 0.75]); iqr = q3 - q1
outliers = df_seq[(df_seq['genome_length_bp'] < q1-1.5*iqr) | (df_seq['genome_length_bp'] > q3+1.5*iqr)]
print(f'Outlier panjang genome: {len(outliers)}')
if len(outliers):
    print(outliers[['assembly_accession','genome_length_bp','isolation_source']].to_string())
plt.tight_layout()
plt.savefig(os.path.join(FIG_OUT, '03_genome_length.png'), dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(df_seq['gc_content']*100, bins=20, color=PALETTE[2], edgecolor='white', alpha=0.85)
axes[0].axvline(df_seq['gc_content'].mean()*100, color='red', ls='--',
                label=f"Mean: {df_seq['gc_content'].mean()*100:.2f}%")
axes[0].axvline(52.0, color='navy', ls=':', label='Referensi LT2: ~52%')
axes[0].set_xlabel('GC Content (%)'); axes[0].set_ylabel('Jumlah Isolat')
axes[0].set_title('Distribusi GC Content'); axes[0].legend()

uniq_src = df_seq['isolation_source'].fillna('other').unique()
cmap_src = {s: PALETTE[i%10] for i,s in enumerate(uniq_src)}
for src in uniq_src:
    mask = df_seq['isolation_source'].fillna('other') == src
    axes[1].scatter(df_seq.loc[mask,'length_Mb'], df_seq.loc[mask,'gc_content']*100,
                    color=cmap_src[src], s=60, alpha=0.8, label=src, edgecolors='white')
axes[1].set_xlabel('Panjang Genome (Mb)'); axes[1].set_ylabel('GC Content (%)')
axes[1].set_title('GC Content vs Panjang Genome'); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_OUT, '03_gc_content.png'), dpi=150); plt.show()

In [ ]:
thr = cfg['genome_qc']['max_n_fraction']
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(df_seq['n_fraction']*100, bins=20, color=PALETTE[3], edgecolor='white', alpha=0.85)
axes[0].axvline(thr*100, color='red', ls='--', label=f'Threshold {thr*100:.0f}%')
axes[0].set_xlabel('N Fraction (%)'); axes[0].set_ylabel('Jumlah Isolat')
axes[0].set_title('Distribusi N Content'); axes[0].legend()

df_sorted = df_seq.sort_values('n_fraction', ascending=False).head(20)
bar_c = ['tomato' if x > thr else PALETTE[3] for x in df_sorted['n_fraction']]
axes[1].bar(range(len(df_sorted)), df_sorted['n_fraction']*100, color=bar_c, edgecolor='white')
axes[1].axhline(thr*100, color='red', ls='--')
axes[1].set_xticks(range(len(df_sorted)))
axes[1].set_xticklabels([a[-10:] for a in df_sorted['assembly_accession']],
                         rotation=45, ha='right', fontsize=7)
axes[1].set_ylabel('N Content (%)'); axes[1].set_title('Top 20 Isolat: N Content Tertinggi')
plt.tight_layout()
plt.savefig(os.path.join(FIG_OUT, '03_n_content.png'), dpi=150); plt.show()
print(f'Isolat N>{thr*100:.0f}% (sudah dibuang di Stage 2): {(~qc_df["qc_pass"]).sum()}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_seq['contig_count'], bins=20, color=PALETTE[5], edgecolor='white', alpha=0.85)
ax.axvline(cfg['genome_qc']['max_contig_count'], color='red', ls='--',
           label=f"Threshold {cfg['genome_qc']['max_contig_count']} contig")
ax.set_xlabel('Jumlah Contig'); ax.set_ylabel('Jumlah Isolat')
ax.set_title('Distribusi Jumlah Contig per Isolat'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_OUT, '03_contig_count.png'), dpi=150); plt.show()
print(f"Median contig: {df_seq['contig_count'].median():.0f}")

---
## Stage 3 — SNP Matrix, Encoding & Distance

**Checkpoints:**
- `artifacts/matrices/snp_matrix.parquet`
- `artifacts/matrices/snp_matrix_encoded.parquet`
- `artifacts/matrices/distance_matrix.csv`

| Langkah | Format |
|---|---|
| SNP matrix (A/T/G/C) | Parquet |
| Encoded SNP (int) | Parquet |
| Pairwise distance | CSV (human-readable) |

In [ ]:
from snp import build_snp_stage3, encode_snp_matrix, compute_distance_matrix
from data import load_genomes_contigs, load_reference_genome

genomes_contigs = load_genomes_contigs(GENOMES_DIR, list(genomes.keys()))
reference_seq   = load_reference_genome(GENOMES_DIR, cfg["data"]["reference_genome"])

snp_df = load_or_compute(
    path=_p(ART["snp_matrix_path"]),
    compute_fn=lambda: build_snp_stage3(cfg, genomes_contigs, reference_seq, ROOT),
    save_fn=save_parquet,
    load_fn=load_parquet,
    force=FORCE_RECOMPUTE,
)

snp_encoded_df = load_or_compute(
    path=_p(ART["snp_encoded_path"]),
    compute_fn=lambda: encode_snp_matrix(snp_df, method="integer"),
    save_fn=save_parquet,
    load_fn=load_parquet,
    force=FORCE_RECOMPUTE,
)

dist_df = load_or_compute(
    path=_p(ART["distance_matrix_path"]),
    compute_fn=lambda: compute_distance_matrix(snp_df),
    save_fn=save_csv_ckpt,
    load_fn=load_csv_ckpt,
    force=FORCE_RECOMPUTE,
)

snp_positions = [int(c) for c in snp_df.columns] if not snp_df.empty else []
print(f"SNP matrix     : {snp_df.shape}  (isolat x posisi SNP)")
print(f"Distance matrix: {dist_df.shape}")
if snp_df.shape[1] > 0:
    print(f"[INFO] Core-SNP posisi variabel: {snp_df.shape[1]:,}")
    print(f"[INFO] Referensi: {cfg['data']['reference_genome']}")


### EDA — SNP Analysis

In [ ]:
if snp_df.empty or snp_df.shape[1] == 0:
    print("[WARN] SNP matrix kosong — visualisasi SNP per isolat dilewati.")
    print(f"  snp_df.shape = {snp_df.shape}")
    print("  Kemungkinan penyebab:")
    print("  1. k-mer alignment menghasilkan sedikit posisi — genome mungkin terlalu divergen")
    print("  2. min_core_fraction terlalu ketat di config.yaml (coba 0.80)")
    print("  3. Dataset terlalu klonal — tidak ada variasi setelah filtering")
else:
    consensus      = snp_df.mode().iloc[0]
    snp_per_isolat = (snp_df != consensus).sum(axis=1).rename("snp_count")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].hist(snp_per_isolat.values, bins=15, color=PALETTE[0], edgecolor="white", alpha=0.85)
    axes[0].axvline(snp_per_isolat.mean(), color="red", ls="--",
                    label=f"Mean: {snp_per_isolat.mean():.0f}")
    axes[0].set_xlabel("Jumlah SNP"); axes[0].set_ylabel("Jumlah Isolat")
    axes[0].set_title("Distribusi SNP per Isolat"); axes[0].legend()

    top20 = snp_per_isolat.sort_values(ascending=False).head(20)
    axes[1].bar(range(len(top20)), top20.values, color=PALETTE[1], edgecolor="white")
    axes[1].set_xticks(range(len(top20)))
    axes[1].set_xticklabels([a[-10:] for a in top20.index], rotation=45, ha="right", fontsize=7)
    axes[1].set_ylabel("Jumlah SNP"); axes[1].set_title("Top 20 Isolat: SNP Terbanyak")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_OUT, "04_snp_count.png"), dpi=150); plt.show()
    print(f"Total posisi SNP : {snp_df.shape[1]}")
    print(f"Per isolat - min:{snp_per_isolat.min()}  median:{snp_per_isolat.median():.0f}  max:{snp_per_isolat.max()}")


In [ ]:
acc_order = [a for a in
             metadata_df.set_index('assembly_accession')['snp_cluster'].fillna('unknown')
             .sort_values().index if a in dist_df.index]
dist_sorted = dist_df.loc[acc_order, acc_order]
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(dist_sorted.values, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='SNP Distance')
ax.set_xticks([]); ax.set_yticks([])
ax.set_title('Pairwise SNP Distance Matrix (diurutkan per SNP cluster)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIG_OUT, '04_distance_heatmap.png'), dpi=150); plt.show()

In [ ]:
n = len(dist_df)
pairs_data = []
for i in range(n):
    for j in range(i+1, n):
        ai, aj = dist_df.index[i], dist_df.index[j]
        ci = acc_to_cluster.get(ai,'unknown'); cj = acc_to_cluster.get(aj,'unknown')
        pairs_data.append({'distance':dist_df.iloc[i,j],'same_cluster':ci==cj,'cluster_i':ci})
df_pairs = pd.DataFrame(pairs_data)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(df_pairs['distance'], bins=40, color=PALETTE[0], edgecolor='white', alpha=0.8)
axes[0].set_xlabel('SNP Distance'); axes[0].set_ylabel('Jumlah Pasangan')
axes[0].set_title('Distribusi Jarak Pairwise')
df_pairs[df_pairs['same_cluster']]['distance'].hist(
    bins=30, ax=axes[1], color='steelblue', alpha=0.7, label='Cluster sama', density=True)
df_pairs[~df_pairs['same_cluster']]['distance'].hist(
    bins=30, ax=axes[1], color='tomato', alpha=0.7, label='Cluster berbeda', density=True)
axes[1].set_xlabel('SNP Distance'); axes[1].set_ylabel('Density')
axes[1].set_title('Jarak: Cluster Sama vs Berbeda'); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_OUT, '04_distance_distribution.png'), dpi=150); plt.show()
sc = df_pairs[df_pairs['same_cluster']]['distance'].mean()
dc = df_pairs[~df_pairs['same_cluster']]['distance'].mean()
print(f'Cluster sama: {sc:.1f} SNP  |  Beda cluster: {dc:.1f} SNP  |  Separation: {dc/max(sc,1):.1f}x')

---
## Stage 3 (lanjutan) — Hierarchical Clustering

In [ ]:
from clustering import hierarchical_clustering, assign_clusters

iso_labels    = dist_df.index.tolist()
linkage_mat   = hierarchical_clustering(dist_df, method=cfg['clustering']['method'])
n_clusters    = min(5, len(iso_labels) - 1)
hclust_labels = assign_clusters(linkage_mat, iso_labels, n_clusters=n_clusters)

print(f'Linkage matrix  : {linkage_mat.shape}')
print(f'Cluster terbentuk: {hclust_labels.nunique()} (target={n_clusters})')
print(hclust_labels.value_counts().to_string())

---
## Baseline Biology 2 — Cluster Validation vs Metadata

Membandingkan hasil genomic clustering (hierarchical) dengan label biologis dari metadata: `isolation_source`, `serovar`, `snp_cluster`, `geo_loc_name`.

| Metrik | Interpretasi |
|---|---|
| ARI (Adjusted Rand Index) | -1..1, nilai 1 = kluster genomik cocok sempurna dengan label metadata |
| NMI (Normalized Mutual Info) | 0..1, ukuran information overlap antar dua pengelompokan |

> Pertanyaan biologis utama: **apakah isolat dari sumber yang sama cenderung berada dalam cluster genomik yang sama?**

In [ ]:
from clustering import validate_clusters_vs_metadata, plot_cluster_composition

# ARI / NMI: seberapa besar genomic cluster berkorelasi dengan label metadata biologis
val_cols = [c for c in ['isolation_source', 'serovar', 'snp_cluster', 'geo_loc_name']
            if c in metadata_df.columns]
cluster_val_df = validate_clusters_vs_metadata(hclust_labels, metadata_df, columns=val_cols)
display(cluster_val_df)

# Stacked bar — komposisi tiap genomic cluster per metadata category (disimpan ke disk)
fig_cl = os.path.join(FIG_OUT, 'clustering')
for col in ['isolation_source', 'serovar']:
    if col in metadata_df.columns:
        plot_cluster_composition(
            hclust_labels, metadata_df, col,
            os.path.join(fig_cl, f'cluster_composition_{col}.png'),
        )

# Inline visualization (baca gambar yang sudah disimpan atau render ulang)
meta_idx = metadata_df.set_index('assembly_accession')
for col in ['isolation_source', 'serovar']:
    if col not in metadata_df.columns:
        continue
    shared = hclust_labels.index.intersection(meta_idx.index)
    df_comp = pd.DataFrame({
        'cluster': hclust_labels.loc[shared].astype(str),
        col: meta_idx.loc[shared, col].fillna('unknown').astype(str),
    })
    ct = df_comp.groupby(['cluster', col]).size().unstack(fill_value=0)
    ct_norm = ct.div(ct.sum(axis=1), axis=0)
    ax = ct_norm.plot(kind='bar', stacked=True, figsize=(max(6, len(ct_norm)*1.4), 4.5),
                      colormap='Set2', edgecolor='white')
    ax.set_title(f'Cluster Composition by {col}', fontsize=12)
    ax.set_xlabel('Genomic Cluster (hierarchical)'); ax.set_ylabel('Proportion')
    ax.legend(title=col, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
from clustering import compute_silhouette
from sklearn.metrics import adjusted_rand_score

# ── E1 — Silhouette Score (SNP distance matrix, precomputed) ─────────────────
sil_e1 = compute_silhouette(dist_df, hclust_labels.values, metric="precomputed")
print(f"[E1] Silhouette Score (SNP distance, hierarchical) : {sil_e1:.4f}")

# ARI: hclust_labels vs snp_cluster NCBI (jika tersedia)
ari_e1 = float("nan")
if "snp_cluster" in metadata_df.columns:
    meta_s = metadata_df.set_index("assembly_accession")
    shared = hclust_labels.index.intersection(meta_s.index)
    h_arr    = hclust_labels.loc[shared].values.astype(str)
    ncbi_arr = meta_s.reindex(shared)["snp_cluster"].fillna("unknown").astype(str).values
    ari_e1 = adjusted_rand_score(ncbi_arr, h_arr)
    print(f"[E1] ARI hclust vs snp_cluster NCBI               : {ari_e1:.4f}")

# ── E1 — Silhouette dari DNABERT embedding dengan hclust labels ──────────────
# Menguji apakah representasi DNABERT preserves genomic cluster structure
sil_e3_clust = float("nan")
try:
    shared_emb = hclust_labels.index.intersection(embedding_df.index)
    sil_e3_clust = compute_silhouette(
        embedding_df.loc[shared_emb],
        hclust_labels.loc[shared_emb].values
    )
    print(f"[E3] Silhouette DNABERT embedding (hclust label)  : {sil_e3_clust:.4f}")
except NameError:
    print("[E3] Silhouette DNABERT: skipped — embedding belum dihasilkan (jalankan Stage 4 dulu)")
except Exception as e:
    print(f"[E3] Silhouette DNABERT: error — {e}")

# ── Within-cluster vs Between-cluster SNP distance boxplot ───────────────────
within_dists, between_dists = [], []
for i in range(len(dist_df)):
    for j in range(i + 1, len(dist_df)):
        d  = float(dist_df.iloc[i, j])
        ai = dist_df.index[i]; aj = dist_df.index[j]
        ci = hclust_labels.get(ai, -1); cj = hclust_labels.get(aj, -1)
        if ci == cj:
            within_dists.append(d)
        else:
            between_dists.append(d)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: boxplot
axes[0].boxplot(
    [within_dists, between_dists],
    labels=["Within cluster", "Between clusters"],
    patch_artist=True,
    boxprops=dict(facecolor=PALETTE[0], alpha=0.6),
    medianprops=dict(color='red', linewidth=2),
)
axes[0].set_ylabel("SNP Distance")
axes[0].set_title("Within vs Between-Cluster SNP Distance\n(Evaluasi Biologis E1)")
w_m, b_m = np.mean(within_dists), np.mean(between_dists)
axes[0].text(0.5, 0.95, f"Separation = {b_m / max(w_m, 1):.1f}×",
             transform=axes[0].transAxes, ha='center', va='top', fontsize=9,
             bbox=dict(boxstyle='round', fc='wheat', alpha=0.7))

# Right: distance by isolation_source (biological grouping)
meta_s = metadata_df.set_index("assembly_accession")
src_within, src_labels = [], []
for src in metadata_df["isolation_source"].dropna().unique():
    accs_src = metadata_df[metadata_df["isolation_source"] == src]["assembly_accession"].tolist()
    accs_src = [a for a in accs_src if a in dist_df.index]
    if len(accs_src) < 2:
        continue
    d_vals = [dist_df.loc[accs_src[i], accs_src[j]]
               for i in range(len(accs_src)) for j in range(i+1, len(accs_src))]
    src_within.append(d_vals)
    src_labels.append(src[:12])

if src_within:
    axes[1].boxplot(src_within, labels=src_labels, patch_artist=True,
                    medianprops=dict(color='red', linewidth=1.5))
    axes[1].set_xticklabels(src_labels, rotation=30, ha='right', fontsize=8)
    axes[1].set_ylabel("SNP Distance (within source)")
    axes[1].set_title("Pairwise SNP Distance\nper Isolation Source")

plt.tight_layout()
plt.savefig(os.path.join(FIG_OUT, 'clustering', '05b_biological_eval.png'), dpi=150)
plt.show()
print(f"\nWithin-cluster mean  : {w_m:.1f} SNP")
print(f"Between-cluster mean : {b_m:.1f} SNP")
print(f"Separation ratio     : {b_m / max(w_m, 1):.2f}×")

### EDA — Hierarchical Clustering

In [ ]:
from scipy.cluster.hierarchy import dendrogram

cluster_ids   = list(set(acc_to_cluster.values()))
cluster_color = {c: plt.cm.tab20(i/max(len(cluster_ids),1)) for i,c in enumerate(cluster_ids)}
fig, ax = plt.subplots(figsize=(16, 5))
dend = dendrogram(linkage_mat, labels=iso_labels, leaf_rotation=90, leaf_font_size=7, ax=ax,
                  color_threshold=0.7*max(linkage_mat[:,2]))
for lbl in ax.get_xticklabels():
    acc = lbl.get_text()
    lbl.set_color(cluster_color.get(acc_to_cluster.get(acc,'unknown'), 'black'))
ax.set_title('Hierarchical Clustering Dendrogram — SNP Distance (Ward)', fontsize=13)
ax.set_ylabel('SNP Distance')
handles = [mpatches.Patch(color=cluster_color[c], label=c[:22]) for c in cluster_ids[:12]]
ax.legend(handles=handles, loc='upper right', fontsize=7, title='SNP Cluster')
plt.tight_layout()
plt.savefig(os.path.join(FIG_OUT, 'clustering', '05_dendrogram.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from scipy.cluster.hierarchy import leaves_list
order_idx      = leaves_list(linkage_mat)
ordered_labels = [iso_labels[i] for i in order_idx]
dist_reordered = dist_df.loc[ordered_labels, ordered_labels]
cluster_series = pd.Series([acc_to_cluster.get(a,'unknown') for a in ordered_labels])
fig = plt.figure(figsize=(11, 9))
ax_hm = fig.add_axes([0.08, 0.08, 0.84, 0.84])
ax_cb = fig.add_axes([0.94, 0.08, 0.02, 0.84])
im = ax_hm.imshow(dist_reordered.values, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, cax=ax_cb, label='SNP Distance')
prev = cluster_series.iloc[0]
for i, clus in enumerate(cluster_series):
    if clus != prev:
        ax_hm.axhline(i-0.5, color='white', lw=1.5); ax_hm.axvline(i-0.5, color='white', lw=1.5)
    prev = clus
ax_hm.set_xticks([]); ax_hm.set_yticks([])
ax_hm.set_title('SNP Distance Heatmap (dendogram order)', fontsize=12)
plt.savefig(os.path.join(FIG_OUT, 'clustering', '05_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as skPCA
import umap as umap_module

if snp_encoded_df.shape[1] == 0:
    print("[SKIP] SNP matrix kosong — PCA/UMAP SNP dilewati. Jalankan ulang setelah SNP matrix berhasil dibangun.")
else:
    snp_vals   = snp_encoded_df.values.astype(float)
    snp_scaled = StandardScaler().fit_transform(snp_vals)
    accs_snp   = snp_encoded_df.index.tolist()
    pca_snp    = skPCA(n_components=10, random_state=42)
    pca_c      = pca_snp.fit_transform(snp_scaled)
    nn         = min(10, len(snp_encoded_df)-1)
    umap_c     = umap_module.UMAP(n_components=2, n_neighbors=nn, random_state=42).fit_transform(snp_scaled)
    print('PCA variance (PC1-3):', [f'{v*100:.1f}%' for v in pca_snp.explained_variance_ratio_[:3]])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    clusters_snp = [acc_to_cluster.get(a,'unknown') for a in accs_snp]
    uniq_cl      = sorted(set(clusters_snp))
    cmap_cl      = {c: PALETTE[i%10] for i,c in enumerate(uniq_cl)}
    for ax_i, (coords, title, xl, yl) in enumerate([
        (pca_c,  'PCA — SNP Matrix',
         f"PC1 ({pca_snp.explained_variance_ratio_[0]*100:.1f}%)",
         f"PC2 ({pca_snp.explained_variance_ratio_[1]*100:.1f}%)"),
        (umap_c, 'UMAP — SNP Matrix', 'UMAP1', 'UMAP2'),
    ]):
        ax = axes[ax_i]
        for cl in uniq_cl:
            mask = np.array([c==cl for c in clusters_snp])
            ax.scatter(coords[mask,0], coords[mask,1], color=cmap_cl[cl],
                       s=70, alpha=0.85, label=cl[:22], edgecolors='white', linewidth=0.5)
        ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(title, fontsize=12); ax.legend(fontsize=7)
    plt.suptitle('Visualisasi Isolat — SNP Matrix', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_OUT, 'clustering', '05_pca_umap_snp.png'), dpi=150)
    plt.show()


---
## Stage 4 — Ekstraksi Window & DNABERT-2 Embedding

**Checkpoint:** `artifacts/embeddings/dnabert_embeddings.npz`

| Langkah | Detail |
|---|---|
| Window type | SNP-context (flank 100 bp) atau sliding |
| Filter N | Buang window dengan N > 10% |
| Cache | `.npz` (float32 terkompresi, jauh lebih kecil dari CSV) |

> **Demo tip:** jika `dnabert_embeddings.npz` sudah ada, cell ini selesai dalam < 1 detik.

In [ ]:
from data import extract_snp_context_windows, extract_windows
from data.preprocess import save_windows

pre_cfg     = cfg['preprocessing']
use_snp_ctx = pre_cfg.get('use_snp_context_windows', True)
max_n_win   = pre_cfg.get('max_n_fraction_window', 0.10)
max_wins    = pre_cfg['max_windows_per_isolate']

if use_snp_ctx and snp_positions:
    print(f"SNP-context windows  flank={pre_cfg['snp_context_flank']} bp  "
          f"max={max_wins}  N<{max_n_win*100:.0f}%")
    windows = extract_snp_context_windows(
        genomes,
        snp_positions = snp_positions,
        flank         = pre_cfg['snp_context_flank'],
        max_windows   = max_wins,
        max_n_frac    = max_n_win,
    )
    save_windows(windows, _p(cfg['data'].get('snp_context_dir', 'data/processed/snp_context/')))
else:
    print(f"Sliding windows  size={pre_cfg['window_size']} bp  max={max_wins}")
    windows = extract_windows(
        genomes,
        window_size = pre_cfg['window_size'],
        max_windows = max_wins,
        max_n_frac  = max_n_win,
    )
    save_windows(windows, _p(cfg['data'].get('windows_dir', 'data/processed/windows/')))

win_counts = {acc: len(ws) for acc, ws in windows.items()}
print(f'Isolat dengan window: {len(win_counts)}')
print(f'Per isolat — min:{min(win_counts.values())}  '
      f'median:{np.median(list(win_counts.values())):.0f}  '
      f'max:{max(win_counts.values())}')

In [ ]:
from embedding import generate_embeddings

# generate_embeddings membaca cfg['artifacts']['embeddings_path'] untuk NPZ cache.
# FORCE_RECOMPUTE dioper via cfg copy agar flag konsisten.
cfg_run = dict(cfg)
cfg_run['pipeline'] = dict(cfg['pipeline'], force_recompute=FORCE_RECOMPUTE)

embedding_df = generate_embeddings(windows, cfg_run)
print(f'\nEmbedding shape: {embedding_df.shape}')
embedding_df.head(3)

In [ ]:
from embedding import generate_stat_embeddings, generate_window_embeddings

# ── Stat embeddings (mean+max+std concat → 2304-dim per isolate) ─────────────
stat_embedding_df = generate_stat_embeddings(windows, cfg_run)
print(f'Stat embedding shape: {stat_embedding_df.shape}')

# ── Per-window embeddings (ragged dict {acc: (n_windows, 768)}) ──────────────
window_embeddings_dict = generate_window_embeddings(windows, cfg_run)
n_wins = [v.shape[0] for v in window_embeddings_dict.values()]
print(f'Window embeddings: {len(window_embeddings_dict)} isolates')
if n_wins:
    print(f'  Windows/isolate: min={min(n_wins)}, max={max(n_wins)}, '
          f'mean={sum(n_wins)/len(n_wins):.1f}')

# Make raw windows accessible to LoRA pipeline inside run_pipeline()
cfg_run['_runtime_windows_dict'] = windows


### EDA — DNABERT-2 Embedding

In [ ]:
norms = np.linalg.norm(embedding_df.values, axis=1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(norms, bins=20, color=PALETTE[4], edgecolor='white', alpha=0.85)
axes[0].set_xlabel('L2 Norm Embedding'); axes[0].set_ylabel('Jumlah Isolat')
axes[0].set_title(f'Distribusi Norm Embedding (mean={norms.mean():.2f})')
sample_dims = embedding_df.iloc[:, :10]
sample_dims.boxplot(ax=axes[1])
axes[1].set_xlabel('Dimensi (0-9)'); axes[1].set_ylabel('Nilai Embedding')
axes[1].set_title('Distribusi Nilai: 10 Dimensi Pertama')
plt.tight_layout()
plt.savefig(os.path.join(FIG_OUT, 'embedding', '06_embedding_stats.png'), dpi=150); plt.show()
print(f'Shape: {embedding_df.shape}  |  DNABERT-2 hidden size = 768')

In [ ]:
if len(embedding_df) >= 5:
    accs_emb     = embedding_df.index.tolist()
    clusters_emb = [acc_to_cluster.get(a,'unknown') for a in accs_emb]
    src_emb      = metadata_df.set_index('assembly_accession')[TARGET_COL]\
                               .reindex(accs_emb).fillna('unknown').tolist()
    pca_e  = skPCA(n_components=2, random_state=42)
    pca_ec = pca_e.fit_transform(embedding_df.values)
    nn_e   = min(5, len(embedding_df)-1)
    umap_ec= umap_module.UMAP(n_components=2, n_neighbors=nn_e, random_state=42).fit_transform(embedding_df.values)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    for ax_i, (coords, title, xu, yu, color_by) in enumerate([
        (pca_ec,  'PCA — DNABERT Embedding',
         f"PC1 ({pca_e.explained_variance_ratio_[0]*100:.1f}%)",
         f"PC2 ({pca_e.explained_variance_ratio_[1]*100:.1f}%)", clusters_emb),
        (umap_ec, 'UMAP — DNABERT Embedding', 'UMAP1', 'UMAP2', src_emb),
    ]):
        ax = axes[ax_i]
        uniq_lbl = sorted(set(color_by))
        cmap_e   = {c: PALETTE[i%10] for i,c in enumerate(uniq_lbl)}
        for lbl in uniq_lbl:
            mask = np.array([c==lbl for c in color_by])
            ax.scatter(coords[mask,0], coords[mask,1], color=cmap_e[lbl],
                       s=70, alpha=0.85, label=str(lbl)[:22], edgecolors='white', linewidth=0.5)
        ax.set_xlabel(xu); ax.set_ylabel(yu)
        ax.set_title(title, fontsize=12); ax.legend(fontsize=7)
    plt.suptitle('Visualisasi DNABERT-2 Embedding', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_OUT, 'embedding', '06_embedding_pca_umap.png'), dpi=150)
    plt.show()
else:
    print('[INFO] Terlalu sedikit isolat untuk PCA/UMAP.')

---
## E5 — K-mer Frequency Features *(opsional, alignment-free)*

**Tetranucleotide frequency** (k=4 → 256 fitur) sebagai baseline alignment-free.

| | K-mer frequency (E5) | DNABERT embedding (E3) |
|---|---|---|
| Representasi | "Bag-of-words" genomik | Context-aware (transformer) |
| Dimensi | 256 | 768 |
| Kecepatan | Sangat cepat | GPU/CPU-heavy |
| Interpretasi | Mudah (% kemunculan) | Sulit (hidden state) |

> Diaktifkan dengan `kmer.enabled: true` di `config.yaml`. Jika dinonaktifkan, cell ini akan melewati komputasi secara otomatis.

In [ ]:
from data import extract_kmer_features

kmer_cfg = cfg.get('kmer', {})
kmer_df  = None

if kmer_cfg.get('enabled', False):
    k = kmer_cfg.get('k', 4)
    print(f"[E5] Menghitung k-mer frequency (k={k}, {4**k} fitur)...")
    kmer_df = extract_kmer_features(genomes, k=k)
    print(f"K-mer DataFrame: {kmer_df.shape}")

    # EDA — distribusi 10 k-mer paling sering
    kmer_mean = kmer_df.mean().sort_values(ascending=False)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    axes[0].bar(range(20), kmer_mean.values[:20],
                color=sns.color_palette('Blues_r', 20), edgecolor='white')
    axes[0].set_xticks(range(20))
    axes[0].set_xticklabels(kmer_mean.index[:20], rotation=45, ha='right', fontsize=8)
    axes[0].set_ylabel('Mean Frequency')
    axes[0].set_title(f'Top 20 {k}-mer Paling Sering (rata-rata lintas isolat)')

    # PCA dari k-mer features
    from sklearn.decomposition import PCA as skPCA2
    pca_km = skPCA2(n_components=2, random_state=42)
    pca_km_c = pca_km.fit_transform(kmer_df.values)
    accs_km  = kmer_df.index.tolist()
    src_km   = metadata_df.set_index('assembly_accession')[TARGET_COL]\
                           .reindex(accs_km).fillna('unknown').tolist()
    uniq_km  = sorted(set(src_km))
    cmap_km  = {s: PALETTE[i % 10] for i, s in enumerate(uniq_km)}
    for s in uniq_km:
        mask = np.array([x == s for x in src_km])
        axes[1].scatter(pca_km_c[mask, 0], pca_km_c[mask, 1],
                        color=cmap_km[s], s=65, alpha=0.85, label=s,
                        edgecolors='white', linewidth=0.5)
    axes[1].set_xlabel(f"PC1 ({pca_km.explained_variance_ratio_[0]*100:.1f}%)")
    axes[1].set_ylabel(f"PC2 ({pca_km.explained_variance_ratio_[1]*100:.1f}%)")
    axes[1].set_title(f'PCA — {k}-mer Frequency Features')
    axes[1].legend(fontsize=8)

    plt.suptitle(f'E5 — Tetranucleotide K-mer (k={k}) Features', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_OUT, 'classification', f'06b_kmer_features.png'), dpi=150)
    plt.show()
else:
    print("[E5] K-mer disabled (kmer.enabled: false in config.yaml).")
    print("     Untuk mengaktifkan: ubah kmer.enabled: true dan FORCE_RECOMPUTE = True")

In [ ]:
# ── AMR Gene Features (E8 / E9 input) ─────────────────────────────────────
# Parse amr_genes column → binary presence/absence matrix per gen unik.
# Digunakan sebagai fitur untuk E8 (AMR-only) dan E9 (SNP+AMR).

from data import extract_amr_features

amr_df = extract_amr_features(metadata_df)

if len(amr_df) > 0:
    print(f'AMR feature matrix: {amr_df.shape}')
    print('Distribusi AMR genes (jumlah isolat yang memiliki gen):')
    print(amr_df.sum().sort_values(ascending=False).to_string())
else:
    print('[AMR] Tidak ada AMR gene features — E8 dan E9 akan diskip.')


---
## Stage 5 — ML Classification (Mode Fitur + AttentionMIL)

**Checkpoints:**
- `artifacts/models/rf_model.joblib`
- `artifacts/reports/metrics.json`
- `artifacts/reports/manifest.json`
- `artifacts/reports/model_comparison.csv`

| Mode | Fitur | Classifier |
|---|---|---|
| `snp_only` | SNP matrix (int-encoded) | RandomForest |
| `dnabert_only` | DNABERT-2 mean-pool 768-dim | RandomForest |
| `hybrid` | SNP + DNABERT concat | RandomForest |
| `kmer_only` *(E5, opsional)* | K-mer frequency k=4 (256-dim) | RandomForest |
| `dnabert_stat_*` | DNABERT mean+max+std (2304-dim) | LR / LinearSVC |
| `dnabert_mil` | DNABERT frozen + **AttentionMIL** (trained per fold) | MIL (PyTorch) |
| `dnabert_lora_mil` *(jika enabled)* | LoRA-DNABERT end-to-end + **AttentionMIL** | MIL (PyTorch) |

> **MIL = Multiple Instance Learning.** Setiap isolat adalah *bag* dari window embeddings;
> AttentionMIL belajar bobot perhatian per-window sehingga posisi genomik kritis mendapat bobot lebih tinggi.


In [ ]:
from models import run_pipeline

model_path   = _p(ART['model_path'])
metrics_path = _p(ART['metrics_path'])

if os.path.exists(model_path) and os.path.exists(metrics_path) and not FORCE_RECOMPUTE:
    print(f'[LOAD]    rf_model.joblib  <- {model_path}')
    best_clf    = load_model(model_path)
    saved       = load_json(metrics_path)
    best_metrics = saved.get('best', {})
    all_metrics  = saved.get('all_modes', {})
else:
    best_clf, best_metrics, all_metrics = run_pipeline(
        embedding_df, metadata_df, cfg_run,
        snp_encoded_df=snp_encoded_df,
        kmer_df=kmer_df,
        amr_df=amr_df if len(amr_df) > 0 else None,
        stat_embedding_df=stat_embedding_df,
        window_embeddings_dict=window_embeddings_dict,
    )
    save_model(best_clf, model_path)
    save_json({'best': best_metrics, 'all_modes': all_metrics}, metrics_path)

print(f'\nBest Balanced Acc: '
      f'{best_metrics.get("balanced_accuracy", best_metrics.get("accuracy", 0)):.4f}  '
      f'Macro F1: {best_metrics.get("f1_macro", best_metrics.get("f1_weighted", 0)):.4f}')


### Experiment Tracking

Layer 1 (selalu aktif): `model_comparison.csv` + `bio_interpretation` per mode di `metrics.json`  
Layer 2 (opt-in): MLflow local — set `tracking.mlflow_enabled: true` di `config.yaml`, lalu `mlflow ui`

In [ ]:
from utils.tracking import enrich_metrics, save_model_comparison, log_mlflow_all_modes

# ── Biological context (computed in earlier cells) ────────────────────────────
bio_ctx = {
    "w_m":    w_m    if 'w_m'    in dir() else float('nan'),
    "b_m":    b_m    if 'b_m'    in dir() else float('nan'),
    "ari_e1": ari_e1 if 'ari_e1' in dir() else float('nan'),
    "sil_e1": sil_e1 if 'sil_e1' in dir() else float('nan'),
}

# ── Layer 1: enrich metrics + save comparison CSV ─────────────────────────────
all_metrics = enrich_metrics(all_metrics, bio_ctx)

# Re-save metrics.json with bio_interpretation included
save_json({'best': best_metrics, 'all_modes': all_metrics}, _p(ART['metrics_path']))

comparison_path = _p(cfg.get('tracking', {}).get(
    'comparison_path', 'artifacts/reports/model_comparison.csv'))
comparison_df = save_model_comparison(
    all_metrics, cfg_run,
    path=comparison_path,
    bio_context=bio_ctx,
)

# ── Display comparison table ──────────────────────────────────────────────────
cols_show = ['experiment', 'feature_set', 'model', 'macro_f1', 'balanced_accuracy',
             'f1_weighted', 'silhouette', 'ari_vs_ncbi', 'separation_ratio',
             'split_type', 'n_train', 'n_test']
display(comparison_df[cols_show].style
        .format({c: '{:.4f}' for c in ['macro_f1','balanced_accuracy','f1_weighted',
                                        'silhouette','ari_vs_ncbi','separation_ratio']},
                na_rep='-')
        .background_gradient(subset=['macro_f1','balanced_accuracy'], cmap='YlGn')
        .set_caption('Model Comparison — E1 to E5'))

# ── Bio interpretation per mode ───────────────────────────────────────────────
print('\n[Bio Interpretation per Mode]')
for mode, m in all_metrics.items():
    bio = m.get('bio_interpretation', '')
    if bio:
        import textwrap
        print(f'\n  [{mode}]')
        print(textwrap.fill(bio, width=88, initial_indent='    ', subsequent_indent='    '))

# ── Layer 2: MLflow (opt-in) ──────────────────────────────────────────────────
log_mlflow_all_modes(
    all_metrics, cfg_run,
    artifact_dir=_p(cfg['output']['figures_dir']),
    bio_context=bio_ctx,
)
if cfg.get('tracking', {}).get('mlflow_enabled', False):
    tracking_uri = cfg.get('tracking', {}).get('tracking_uri', 'mlruns')
    print(f'\n[MLFLOW] View UI: mlflow ui --backend-store-uri {_p(tracking_uri)}')

### AttentionMIL — Interpretasi Bobot Perhatian Window

AttentionMIL mempelajari bobot `α_i` per window. Window dengan bobot tinggi mengandung
sinyal genomik yang paling diskriminatif untuk source attribution.
Visualisasi di bawah menampilkan distribusi bobot dan top-attended windows.


In [ ]:
# AttentionMIL — visualisasi bobot perhatian per isolat
import torch
from embedding.mil import AttentionMIL, predict_mil, get_top_windows

mil_key = 'dnabert_mil'
if mil_key in all_metrics and window_embeddings_dict:
    print(f'[MIL] Macro F1 = {all_metrics[mil_key].get("f1_macro", 0):.4f}  '
          f'Balanced Acc = {all_metrics[mil_key].get("balanced_accuracy", 0):.4f}')

    # Ambil bobot perhatian untuk beberapa isolat representatif
    sample_accs = list(window_embeddings_dict.keys())[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 6))
    axes = axes.flatten()

    for ax_i, acc in enumerate(sample_accs):
        embs = torch.tensor(window_embeddings_dict[acc], dtype=torch.float32)
        # Gunakan model MIL jika tersimpan di all_metrics, else hanya visualisasi uniform
        mil_model_obj = all_metrics[mil_key].get('_mil_model_ref')
        if mil_model_obj is not None:
            with torch.no_grad():
                _, weights = mil_model_obj(embs)
            weights_np = weights.cpu().numpy()
        else:
            weights_np = (1.0 / len(embs)) * np.ones(len(embs))

        ax = axes[ax_i]
        ax.bar(range(len(weights_np)), weights_np, color=PALETTE[ax_i % len(PALETTE)], alpha=0.8)
        ax.set_title(f'{acc[:20]}', fontsize=8)
        ax.set_xlabel('Window index')
        ax.set_ylabel('Attention weight')
        ax.grid(axis='y', alpha=0.3)

    for ax_i in range(len(sample_accs), len(axes)):
        axes[ax_i].set_visible(False)

    plt.suptitle('AttentionMIL — Bobot Perhatian per Window (per isolat)', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_OUT, 'classification', '08_mil_attention_weights.png'), dpi=150)
    plt.show()
else:
    print('[MIL] dnabert_mil belum dijalankan atau window_embeddings_dict kosong.')
    print('  Pastikan DNABERT embedding stage sudah lengkap dan run_pipeline() dipanggil dengan window_embeddings_dict.')


### EDA — ML Evaluation

In [ ]:
import math

modes_valid = {k: v for k, v in all_metrics.items() if v.get('balanced_accuracy', v.get('accuracy', 0)) > 0}
if len(modes_valid) > 0:
    df_res = pd.DataFrame([{
        'Mode':         k,
        'Macro F1':     v.get('f1_macro', 0.0),
        'Balanced Acc': v.get('balanced_accuracy', 0.0),
        'F1 (weighted)':v.get('f1_weighted', 0.0),
        'Silhouette':   v.get('silhouette', float('nan')),
        'Split':        v.get('split_type', '-'),
    } for k, v in modes_valid.items()])
    display(df_res)

    # Bar chart — Macro F1 + Balanced Accuracy (metrik utama untuk class-imbalance)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    x = np.arange(len(df_res)); w = 0.28
    b1 = ax.bar(x - w,   df_res['Macro F1'],     w, label='Macro F1',     color=PALETTE[0])
    b2 = ax.bar(x,       df_res['Balanced Acc'], w, label='Balanced Acc', color=PALETTE[1])
    b3 = ax.bar(x + w,   df_res['F1 (weighted)'],w, label='F1 (weighted)',color=PALETTE[2], alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(df_res['Mode'])
    ax.set_ylim(0, 1.2); ax.set_ylabel('Score')
    _title_modes = 'E2 SNP-only · E3 DNABERT · E4 Hybrid'
    if kmer_df is not None:
        _title_modes += ' · E5 K-mer'
    ax.set_title(f'Ablation Study — {_title_modes}\n(Evaluasi Source Attribution)')
    ax.legend(loc='upper right', fontsize=9); ax.grid(axis='y', alpha=0.3)
    for bar in list(b1) + list(b2) + list(b3):
        h = bar.get_height()
        if h > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                    f'{h:.2f}', ha='center', va='bottom', fontsize=7.5)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_OUT, 'classification', '07_ablation_study.png'), dpi=150)
    plt.show()
else:
    print('[INFO] Belum ada mode yang berhasil dijalankan.')

In [ ]:
for mode, m in all_metrics.items():
    if m.get('accuracy', 0) == 0:
        continue
    print(f'\n=== {mode} ===')
    print(f"Accuracy: {m['accuracy']:.4f}  |  F1-weighted: {m['f1_weighted']:.4f}")
    if m.get('report'):
        print(m['report'])

---
### Leakage Assessment — Group-aware vs Naive Split

Membandingkan hasil evaluasi dengan dua skenario split:

| Skenario | Split | Tujuan |
|---|---|---|
| **Group-aware** (utama) | `StratifiedGroupKFold` berdasarkan `snp_cluster` | Hasil resmi yang dilaporkan |
| **Naive** (pembanding) | `StratifiedShuffleSplit` biasa | Cek apakah ada overestimasi akibat leakage |

> Jika naive F1 >> group-aware F1, artinya isolat berkerabat dekat (satu SNP cluster) tersebar ke train dan test, menyebabkan **data leakage** dan overestimasi performa nyata.

In [ ]:
import math

# Naive split — jalankan tanpa group constraint (hanya untuk perbandingan)
# CATATAN: hasilnya TIDAK disimpan ke checkpoint; hanya ditampilkan di sini.
print('Menjalankan naive split untuk leakage assessment...')
_, _, naive_metrics = run_pipeline(
    embedding_df, metadata_df, cfg_run,
    snp_encoded_df=snp_encoded_df,
    kmer_df=kmer_df,
    amr_df=amr_df if len(amr_df) > 0 else None,
    stat_embedding_df=stat_embedding_df,
    window_embeddings_dict=window_embeddings_dict,
    use_groups=False,
)

# Tabel perbandingan: Group-aware vs Naive
def _m(d, k): return d.get(k, float('nan'))

# Sertakan semua mode aktif termasuk MIL
compare_modes = ['snp_only', 'dnabert_only', 'hybrid']
if kmer_df is not None:
    compare_modes.append('kmer_only')
for _extra in ['snp_lr', 'snp_svc', 'amr_lr', 'snp_amr_lr', 'dnabert_mil', 'dnabert_lora_mil']:
    if all_metrics.get(_extra, {}).get('f1_macro', 0) > 0:
        compare_modes.append(_extra)

rows_cmp = []
for mode in compare_modes:
    ga = all_metrics.get(mode, {})
    nv = naive_metrics.get(mode, {})
    if not ga and not nv:
        continue
    delta_f1  = _m(nv, 'f1_macro') - _m(ga, 'f1_macro')
    delta_acc = _m(nv, 'balanced_accuracy') - _m(ga, 'balanced_accuracy')
    rows_cmp.append({
        'Mode':                mode,
        'GA Macro F1':         _m(ga, 'f1_macro'),
        'GA Bal. Acc':         _m(ga, 'balanced_accuracy'),
        'GA Split':            _m(ga, 'split_type'),
        'Naive Macro F1':      _m(nv, 'f1_macro'),
        'Naive Bal. Acc':      _m(nv, 'balanced_accuracy'),
        'Naive Split':         _m(nv, 'split_type'),
        'DeltaF1 (naive-GA)':      delta_f1,
        'DeltaBalAcc (naive-GA)': delta_acc,
    })

df_cmp = pd.DataFrame(rows_cmp)
display(df_cmp)

# Interpretasi otomatis
print('\n[Interpretasi Leakage Assessment]')
for row in rows_cmp:
    df1  = row['DeltaF1 (naive-GA)']
    mode = row['Mode']
    if math.isnan(df1):
        continue
    if df1 > 0.15:
        print(f'  {mode}: DeltaF1={df1:+.3f}  - TINGGI: kemungkinan leakage dari SNP cluster.')
        print(f'    Hasil group-aware ({row["GA Macro F1"]:.3f}) adalah estimasi lebih realistis.')
    elif df1 > 0.05:
        print(f'  {mode}: DeltaF1={df1:+.3f}  - MODERAT: sebagian isolat berkerabat masuk test.')
    else:
        print(f'  {mode}: DeltaF1={df1:+.3f}  - RENDAH: leakage minimal, split group efektif.')

# Bar chart: Group-aware vs Naive F1
if len(df_cmp) > 0:
    fig, ax = plt.subplots(figsize=(max(9, len(df_cmp)*2.5), 4.5))
    x = np.arange(len(df_cmp)); w = 0.35
    b1 = ax.bar(x - w/2, df_cmp['GA Macro F1'],    w, label='Group-aware (utama)', color=PALETTE[0])
    b2 = ax.bar(x + w/2, df_cmp['Naive Macro F1'], w, label='Naive (perbandingan)', color='tomato', alpha=0.75)
    ax.set_xticks(x); ax.set_xticklabels(df_cmp['Mode'], rotation=25, ha='right')
    ax.set_ylim(0, 1.2); ax.set_ylabel('Macro F1-score')
    ax.set_title('Leakage Assessment: Group-aware vs Naive Split\n'
                 '(selisih besar = overestimasi akibat SNP cluster leakage)')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    for bar in list(b1) + list(b2):
        h = bar.get_height()
        if h > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                    f'{h:.2f}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_OUT, 'classification', '07b_leakage_assessment.png'), dpi=150)
    plt.show()


---
### Learning Curve — Apakah Dataset Perlu Ditambah?

Learning curve mendiagnosis apakah performa **test F1 masih naik** seiring bertambahnya
ukuran training set:

| Pola | Interpretasi |
|---|---|
| Test F1 masih naik → plateau belum tercapai | **Tambah data** kemungkinan besar membantu |
| Test F1 sudah flat, train F1 jauh lebih tinggi | **Overfitting** — coba regularisasi / fitur lebih sedikit |
| Keduanya flat dan rendah | **High bias** — fitur / model perlu diperbaiki |

Evaluasi menggunakan `StratifiedGroupKFold` (group-aware, sama seperti evaluasi utama).
Mode yang dianalisis: **kmer_only** (alignment-free, representatif) dan **snp_amr** (best balanced-acc).

In [ ]:
from models import run_learning_curve
from models.pipeline import _extract_groups
from models.trainer import _join_target, get_sample_index

lc_results = {}

# ── Helper: build groups array ─────────────────────────────────────────────
def _lc_groups(feature_df):
    idx = get_sample_index(feature_df, metadata_df, TARGET_COL)
    return _extract_groups(idx, metadata_df, 'snp_cluster')

# ── E5 kmer_only ───────────────────────────────────────────────────────────
if kmer_df is not None and len(kmer_df) > 0:
    print('=== Learning Curve: kmer_only ===')
    X_lc, y_lc, _ = _join_target(kmer_df, metadata_df, TARGET_COL)
    lc_results['kmer'] = run_learning_curve(
        X_lc, y_lc, _lc_groups(kmer_df), cfg_run,
        out_dir=os.path.join(FIG_OUT, 'classification'),
        mode_name='kmer',
    )
else:
    print('[SKIP] kmer_df tidak tersedia — aktifkan kmer.enabled di config.yaml')

# ── SNP + AMR combined (best balanced-acc mode) ────────────────────────────
if snp_encoded_df is not None and len(snp_encoded_df) > 0 and len(amr_df) > 0:
    print('=== Learning Curve: snp_amr ===')
    shared = snp_encoded_df.index.intersection(amr_df.index)
    if len(shared) >= 8:
        combined = snp_encoded_df.loc[shared].join(amr_df.loc[shared], how='inner')
        X_lc2, y_lc2, _ = _join_target(combined, metadata_df, TARGET_COL)
        lc_results['snp_amr'] = run_learning_curve(
            X_lc2, y_lc2, _lc_groups(combined), cfg_run,
            out_dir=os.path.join(FIG_OUT, 'classification'),
            mode_name='snp_amr',
        )

# ── Interpretasi otomatis ──────────────────────────────────────────────────
print('[Ringkasan Learning Curve]')
for mode, lc in lc_results.items():
    test_scores = lc['test_scores_mean']
    train_scores = lc['train_scores_mean']
    delta = test_scores[-1] - test_scores[0]
    gap   = train_scores[-1] - test_scores[-1]
    print(f'  [{mode}]  CV={lc["cv"]}')
    print(f'    Train sizes  : {lc["train_sizes"]}')
    print(f'    Train F1     : {[f"{v:.3f}" for v in train_scores]}')
    print(f'    Test  F1     : {[f"{v:.3f}" for v in test_scores]}')
    print(f'    Delta test   : {delta:+.3f}  |  Train-test gap (final): {gap:.3f}')
    if delta > 0.05:
        print(f'    → Test F1 masih naik (+{delta:.3f}): penambahan isolat kemungkinan MEMBANTU.')
    elif delta < -0.03:
        print(f'    → Test F1 turun ({delta:.3f}): mungkin variance tinggi — perlu regularisasi.')
    else:
        print(f'    → Test F1 relatif flat: bottleneck bukan hanya jumlah data.')
    if gap > 0.25:
        print(f'    → Train-test gap {gap:.3f} > 0.25: overfitting — model terlalu kompleks atau data kecil.')

if lc_results:
    print(f'Figure disimpan di: {os.path.join(FIG_OUT, "classification")}')
else:
    print('[SKIP] Tidak ada mode yang dapat dijalankan — pastikan kmer.enabled=true di config.yaml')

---
## Forensic Interpretation Summary

In [ ]:
sc_mean = df_pairs[df_pairs['same_cluster']]['distance'].mean()
dc_mean = df_pairs[~df_pairs['same_cluster']]['distance'].mean()

print('='*60)
print('  RINGKASAN FORENSIK GENOMIK — SalmoTrace-BERT')
print('='*60)
print(f'\n[1] DATASET')
print(f'    Serovar       : {metadata_df["serovar"].value_counts().index[0]}')
print(f'    Isolat valid  : {len(metadata_df)}')
print(f'    Referensi     : {cfg["data"]["reference_genome"]}')
print(f'\n[2] KUALITAS SEKUENS')
print(f'    Panjang median : {df_seq["genome_length_bp"].median()/1e6:.3f} Mb')
print(f'    GC content     : {df_seq["gc_content"].mean()*100:.2f}% ± {df_seq["gc_content"].std()*100:.2f}%')
print(f'    Isolat dibuang : {(~qc_df["qc_pass"]).sum()} (gagal QC)')
print(f'\n[3] SNP ANALYSIS')
print(f'    Posisi SNP     : {snp_df.shape[1]}')
print(f'    Jarak maks     : {int(dist_df.values.max())} SNP')
print(f'    Sama-cluster (NCBI snp_cluster) : {sc_mean:.1f} SNP  |  Beda-cluster: {dc_mean:.1f} SNP')
print(f'    Separation ratio                : {dc_mean/max(sc_mean,1):.1f}x (berdasarkan NCBI snp_cluster)')
print(f'\n[4] CLUSTERING')
print(f'    SNP cluster unik: {metadata_df["snp_cluster"].nunique()}')
print(f'    Hclust hasil    : {hclust_labels.nunique()} kluster (Ward)')
print(f'\n[5] ML CLASSIFICATION — Target: {TARGET_COL}')
for mode, m in all_metrics.items():
    if m.get('accuracy', 0) > 0:
        print(f'    {mode:<25} Acc={m["accuracy"]:.3f}  F1={m["f1_weighted"]:.3f}')
print('='*60)

In [ ]:
print('TOP 10 PASANGAN ISOLAT DENGAN JARAK SNP TERKECIL (relatif paling mirip dalam dataset):')
print('-'*60)
for _, row in df_pairs.sort_values('distance').head(10).iterrows():
    sc_label = 'SAMA' if row['same_cluster'] else 'BEDA'
    print(f'  dist={int(row["distance"]):>5}  cluster={sc_label}  ({row["cluster_i"][:30]})')

print()
print('CATATAN INTERPRETASI:')
print('-'*60)
_max_dist    = int(dist_df.values.max())
_min_nonzero = int(dist_df.values[dist_df.values > 0].min())
print(f'Dalam dataset ini, jarak SNP berkisar antara {_min_nonzero} hingga {_max_dist} SNP.')
print('Pasangan dengan jarak terkecil menunjukkan kedekatan relatif dalam dataset,')
print('bukan indikasi common source tanpa validasi epidemiologis.')
print('NCBI Pathogen Detection mendefinisikan SNP cluster berdasarkan pipeline independen.')

print('TOP 5 ISOLAT PALING TERISOLASI (potensial strain baru):')
print('-'*60)
meta_idx  = metadata_df.set_index('assembly_accession')
min_dists = dist_df.apply(lambda col: col[col>0].min(), axis=0).sort_values(ascending=False)
for acc, d in min_dists.head(5).items():
    src = meta_idx.loc[acc, 'isolation_source'] if acc in meta_idx.index else '?'
    cl  = acc_to_cluster.get(acc, '?')
    print(f'  {acc}  min_dist={int(d):>5}  src={src}  cluster={cl}')

In [ ]:
from report import generate_report, build_forensic_table, generate_forensic_summary

cluster_result = {'n_clusters': int(hclust_labels.nunique())}

# Summary report — inclui tabel perbandingan E2/E3/E4 dan validasi cluster biologis
generate_report(
    metadata_df, snp_df, dist_df, cluster_result, best_metrics, cfg,
    all_metrics=all_metrics,
    cluster_validation_df=cluster_val_df,
)

# Forensic interpretation — per-isolat: nearest neighbor, SNP dist, source prediction
if best_clf is not None:
    from models.pipeline import _select_forensic_feature_df
    _best_mode = best_metrics.get('best_mode')
    _forensic_feat = _select_forensic_feature_df(
        _best_mode, embedding_df, snp_encoded_df, kmer_df,
        amr_df if len(amr_df) > 0 else None,
    )
    forensic_df = build_forensic_table(
        dist_df, metadata_df, target_col=TARGET_COL,
        clf=best_clf, feature_df=_forensic_feat,
    )
    generate_forensic_summary(
        forensic_df, all_metrics,
        _p(ART.get('forensic_path', 'artifacts/reports/forensic_report.txt')),
    )
    print()
    display(forensic_df[['isolation_source', 'nearest_neighbor',
                          'snp_distance_to_nearest', 'nearest_source',
                          'predicted_source', 'prediction_confidence']].head(10))

save_manifest(
    cfg, metadata_df, snp_df, all_metrics,
    _p(ART['manifest_path']),
)
print('\nPipeline selesai.')
print(f'Artifacts tersimpan di: {_p("artifacts/")}')

---
### Ringkasan Forensik Genomik

Berdasarkan hasil pairwise SNP distance, isolat-isolat yang berada dalam kelompok yang sama menunjukkan jarak SNP lebih kecil dibanding isolat dari kelompok berbeda. Hal ini mengindikasikan kedekatan genomik yang lebih tinggi dan kemungkinan berasal dari *lineage* yang lebih berkerabat. Dalam konteks forensik genomik, kedekatan ini dapat digunakan sebagai dasar awal untuk menelusuri kemungkinan hubungan antar-isolat pada rantai kontaminasi pangan.

Hasil klasifikasi *source attribution* menunjukkan bahwa pada konfigurasi dataset saat ini, fitur SNP maupun embedding DNABERT **belum mampu menangkap sinyal sumber isolasi secara stabil**. Macro F1 yang rendah dan balanced accuracy yang hampir setara pada semua mode mengindikasikan bahwa dataset terlalu kecil, label terlalu granular, dan sinyal genomik belum cukup terpisah. Ini merupakan **hasil negatif yang jujur** dan perlu dilaporkan sebagai temuan utama, bukan sebagai keterbatasan minor.

**Poin kunci:**
1. Clustering SNP memberikan pemisahan yang sangat lemah — separation ratio mendekati 1.0×, artinya isolat dalam dan antar-kluster hampir sama jauhnya
2. Label `isolation_source` ternormalisasi sebelum training — mencegah bias dari variasi penulisan
3. SNP-context window — DNABERT mendapat daerah flanking SNP yang paling informatif filogenetik
4. Interpretasi biologis tabel E1–E5 ada di sel di atas — bacalah angka bersama maknanya
5. Analisis ini adalah **simulasi forensik genomik** — bukan pembuktian epidemiologis langsung

---
*SalmoTrace-BERT · Data: NCBI Pathogen Detection · DNABERT-2: `zhihan1996/DNABERT-2-117M`*

In [ ]:
import math

# TABEL EVALUASI — E0 s/d E13 + MIL

def _fmt(v, fmt='.4f'):
    if isinstance(v, float) and math.isnan(v): return '  -   '
    return format(v, fmt) if v is not None else '  -   '

sil_e2 = all_metrics.get('snp_only',     {}).get('silhouette', float('nan'))
sil_e3 = all_metrics.get('dnabert_only', {}).get('silhouette', float('nan'))
sil_e4 = all_metrics.get('hybrid',       {}).get('silhouette', float('nan'))
sil_e5 = all_metrics.get('kmer_only',    {}).get('silhouette', float('nan'))

rows = [
    ('E1  SNP Clustering',  'SNP distance',      'Hierarchical',
     '   -  ', '   -  ', _fmt(sil_e1), _fmt(ari_e1)),
    ('E0  Dummy',           'SNP matrix',        'DummyClassifier',
     _fmt(all_metrics.get('dummy',{}).get('f1_macro', float('nan'))),
     _fmt(all_metrics.get('dummy',{}).get('balanced_accuracy', float('nan'))),
     '   -  ', '   -  '),
    ('E2  SNP-only RF',     'SNP matrix (int)',  'RandomForest',
     _fmt(all_metrics.get('snp_only',{}).get('f1_macro', float('nan'))),
     _fmt(all_metrics.get('snp_only',{}).get('balanced_accuracy', float('nan'))),
     _fmt(sil_e2), '   -  '),
    ('E3  DNABERT RF',      'DNABERT embed.',    'RandomForest',
     _fmt(all_metrics.get('dnabert_only',{}).get('f1_macro', float('nan'))),
     _fmt(all_metrics.get('dnabert_only',{}).get('balanced_accuracy', float('nan'))),
     _fmt(sil_e3), '   -  '),
    ('E4  Hybrid RF',       'SNP + DNABERT',     'RandomForest',
     _fmt(all_metrics.get('hybrid',{}).get('f1_macro', float('nan'))),
     _fmt(all_metrics.get('hybrid',{}).get('balanced_accuracy', float('nan'))),
     _fmt(sil_e4), '   -  '),
]

# E5
if all_metrics.get('kmer_only'):
    rows.append(('E5  K-mer RF', f'{cfg.get("kmer",{}).get("k",4)}-mer freq',
                 'RandomForest',
                 _fmt(all_metrics['kmer_only'].get('f1_macro', float('nan'))),
                 _fmt(all_metrics['kmer_only'].get('balanced_accuracy', float('nan'))),
                 _fmt(sil_e5), '   -  '))

# E6
if all_metrics.get('snp_lr'):
    rows.append(('E6  SNP LR', 'SNP+SelectKBest',
                 'LogisticReg(bal)',
                 _fmt(all_metrics['snp_lr'].get('f1_macro', float('nan'))),
                 _fmt(all_metrics['snp_lr'].get('balanced_accuracy', float('nan'))),
                 '   -  ', '   -  '))

# E7
if all_metrics.get('snp_svc'):
    rows.append(('E7  SNP LinearSVC', 'SNP+SelectKBest',
                 'LinearSVC(bal)',
                 _fmt(all_metrics['snp_svc'].get('f1_macro', float('nan'))),
                 _fmt(all_metrics['snp_svc'].get('balanced_accuracy', float('nan'))),
                 '   -  ', '   -  '))

# E8
if all_metrics.get('amr_lr'):
    rows.append(('E8  AMR-only LR', 'AMR genes (binary)',
                 'LogisticReg(bal)',
                 _fmt(all_metrics['amr_lr'].get('f1_macro', float('nan'))),
                 _fmt(all_metrics['amr_lr'].get('balanced_accuracy', float('nan'))),
                 '   -  ', '   -  '))

# E9
if all_metrics.get('snp_amr_lr'):
    rows.append(('E9  SNP+AMR LR', 'SNP+AMR+SelectKBest',
                 'LogisticReg(bal)',
                 _fmt(all_metrics['snp_amr_lr'].get('f1_macro', float('nan'))),
                 _fmt(all_metrics['snp_amr_lr'].get('balanced_accuracy', float('nan'))),
                 '   -  ', '   -  '))

# E3b
if all_metrics.get('dnabert_lr'):
    rows.append(('E3b DNABERT LR', 'DNABERT embed.', 'LogisticReg(bal)',
                 _fmt(all_metrics['dnabert_lr'].get('f1_macro', float('nan'))),
                 _fmt(all_metrics['dnabert_lr'].get('balanced_accuracy', float('nan'))),
                 '   -  ', '   -  '))

# E3c
if all_metrics.get('dnabert_svc'):
    rows.append(('E3c DNABERT SVC', 'DNABERT embed.', 'LinearSVC(bal)',
                 _fmt(all_metrics['dnabert_svc'].get('f1_macro', float('nan'))),
                 _fmt(all_metrics['dnabert_svc'].get('balanced_accuracy', float('nan'))),
                 '   -  ', '   -  '))

# E10
if all_metrics.get('kmer_amr_lr'):
    rows.append(('E10 kmer+AMR LR', 'kmer+AMR', 'LogisticReg(bal)',
                 _fmt(all_metrics['kmer_amr_lr'].get('f1_macro', float('nan'))),
                 _fmt(all_metrics['kmer_amr_lr'].get('balanced_accuracy', float('nan'))),
                 '   -  ', '   -  '))

# E11
if all_metrics.get('kmer_amr_svc'):
    rows.append(('E11 kmer+AMR SVC', 'kmer+AMR', 'LinearSVC(bal)',
                 _fmt(all_metrics['kmer_amr_svc'].get('f1_macro', float('nan'))),
                 _fmt(all_metrics['kmer_amr_svc'].get('balanced_accuracy', float('nan'))),
                 '   -  ', '   -  '))

# E12
if all_metrics.get('snp_kmer_amr_lr'):
    rows.append(('E12 SNP+kmer+AMR LR', 'SNP+kmer+AMR+Sel', 'LogisticReg(bal)',
                 _fmt(all_metrics['snp_kmer_amr_lr'].get('f1_macro', float('nan'))),
                 _fmt(all_metrics['snp_kmer_amr_lr'].get('balanced_accuracy', float('nan'))),
                 '   -  ', '   -  '))

# E13
if all_metrics.get('snp_kmer_amr_rf'):
    rows.append(('E13 SNP+kmer+AMR RF', 'SNP+kmer+AMR', 'RandomForest(bal)',
                 _fmt(all_metrics['snp_kmer_amr_rf'].get('f1_macro', float('nan'))),
                 _fmt(all_metrics['snp_kmer_amr_rf'].get('balanced_accuracy', float('nan'))),
                 '   -  ', '   -  '))

# E_MIL — AttentionMIL (frozen DNABERT)
if all_metrics.get('dnabert_mil'):
    rows.append(('E_MIL  DNABERT+MIL', 'Per-window DNABERT', 'AttentionMIL',
                 _fmt(all_metrics['dnabert_mil'].get('f1_macro', float('nan'))),
                 _fmt(all_metrics['dnabert_mil'].get('balanced_accuracy', float('nan'))),
                 '   -  ', '   -  '))

# E_LoRA — LoRA-DNABERT + AttentionMIL (end-to-end)
if all_metrics.get('dnabert_lora_mil'):
    rows.append(('E_LoRA DNABERT+LoRA', 'LoRA-DNABERT+MIL', 'AttentionMIL (e2e)',
                 _fmt(all_metrics['dnabert_lora_mil'].get('f1_macro', float('nan'))),
                 _fmt(all_metrics['dnabert_lora_mil'].get('balanced_accuracy', float('nan'))),
                 '   -  ', '   -  '))

hdr = f"{'Eksperimen':<22} {'Fitur':<21} {'Model':<18} {'Macro F1':>9} {'Bal. Acc':>9} {'Silhouette':>11} {'ARI':>8}"
sep = '-' * len(hdr)
print('\n' + '=' * len(hdr))
print('  TABEL EVALUASI SalmoTrace-BERT (E0-E13 + MIL)')
print('=' * len(hdr))
print(hdr)
print(sep)
for r in rows:
    print(f"  {r[0]:<20} {r[1]:<21} {r[2]:<18} {r[3]:>9} {r[4]:>9} {r[5]:>11} {r[6]:>8}")
print('=' * len(hdr))

# Interpretasi otomatis
print('\n[Interpretasi Biologis]')
if not math.isnan(sil_e1):
    if sil_e1 > 0.5:   print(f'  E1 Silhouette={sil_e1:.3f}: cluster genomik terpisah dengan baik.')
    elif sil_e1 > 0.2:  print(f'  E1 Silhouette={sil_e1:.3f}: cluster genomik cukup terpisah.')
    else:               print(f'  E1 Silhouette={sil_e1:.3f}: cluster genomik tumpang tindih.')
if not math.isnan(ari_e1):
    if ari_e1 > 0.5:    print(f'  ARI={ari_e1:.3f}: cluster genomik sangat sesuai dengan SNP cluster NCBI.')
    elif ari_e1 > 0.1:  print(f'  ARI={ari_e1:.3f}: kesesuaian rendah hingga parsial.')
    else:               print(f'  ARI={ari_e1:.3f}: cluster genomik belum sesuai dengan SNP cluster NCBI.')

print('\n[Interpretasi Klasifikasi]')
modes_res = {k: v for k, v in all_metrics.items()
             if v.get('f1_macro', 0) > 0 and k != 'dummy'}
dummy_f1 = all_metrics.get('dummy', {}).get('f1_macro', 0)
print(f'  Dummy baseline Macro F1 = {dummy_f1:.4f}')
if modes_res:
    best_f1    = max(modes_res[k].get('f1_macro', 0) for k in modes_res)
    best_modes = [k for k in modes_res if abs(modes_res[k].get('f1_macro', 0) - best_f1) < 1e-6]
    best_mode  = best_modes[0]
    if best_f1 <= dummy_f1 + 0.02:
        print('  [WARN] Semua model hampir setara dengan DummyClassifier!')
        print('         Fitur belum cukup informatif atau data terlalu sedikit.')
    else:
        print(f'  Mode terbaik: {best_mode}  (Macro F1 = {best_f1:.4f})')
    e2_f1  = modes_res.get('snp_only',     {}).get('f1_macro', 0)
    e6_f1  = modes_res.get('snp_lr',       {}).get('f1_macro', 0)
    e7_f1  = modes_res.get('snp_svc',      {}).get('f1_macro', 0)
    e3_f1  = modes_res.get('dnabert_only', {}).get('f1_macro', 0)
    mil_f1 = modes_res.get('dnabert_mil',  {}).get('f1_macro', 0)
    lora_f1= modes_res.get('dnabert_lora_mil', {}).get('f1_macro', 0)
    if max(e6_f1, e7_f1) > e2_f1:
        print(f'  LR/SVC (E6/E7) F1={max(e6_f1,e7_f1):.3f} > RF E2 F1={e2_f1:.3f}: '
              f'model linear lebih baik untuk dataset kecil.')
    if e3_f1 < e2_f1:
        print(f'  DNABERT (E3) F1={e3_f1:.3f} < SNP (E2) F1={e2_f1:.3f}: '
              f'SNP lebih stabil dari embedding pada dataset ini.')
    if mil_f1 > e3_f1 and mil_f1 > 0:
        print(f'  AttentionMIL F1={mil_f1:.3f} > DNABERT-RF F1={e3_f1:.3f}: '
              f'attention pooling lebih efektif dari mean pooling.')
    if lora_f1 > mil_f1 and lora_f1 > 0:
        print(f'  LoRA-MIL F1={lora_f1:.3f} > AttentionMIL F1={mil_f1:.3f}: '
              f'fine-tuning DNABERT meningkatkan performa.')
    amr_f1 = modes_res.get('amr_lr', {}).get('f1_macro', 0)
    if amr_f1 > 0:
        verdict = 'informatif' if amr_f1 > dummy_f1 + 0.05 else 'kurang informatif sendiri'
        print(f'  AMR-only (E8) F1={amr_f1:.3f}: AMR genes {verdict}.')


---
## Interpretasi Biologis

> Setiap angka keluaran program diterjemahkan ke makna biologis.  
> Analisis ini merupakan **simulasi forensik genomik** menggunakan data publik — bukan pembuktian epidemiologis langsung.

### Kerangka Biologis

Dasar interpretasi seluruh eksperimen adalah prinsip *whole-genome sequencing* (WGS) untuk investigasi wabah:

> **Semakin kecil jarak SNP antar-isolat, semakin tinggi kedekatan genomiknya**, sehingga isolat-isolat tersebut lebih mungkin berasal dari garis keturunan atau sumber kontaminasi yang berkerabat. Sebaliknya, jarak SNP besar menunjukkan perbedaan genomik yang lebih jauh dan dapat mengindikasikan sumber atau *lineage* yang berbeda.

SNP (*Single Nucleotide Polymorphism*) adalah variasi nukleotida tunggal yang timbul akibat mutasi (*substitution*, *insertion*, *deletion*). Akumulasi SNP antar-isolat mencerminkan divergensi evolusioner — makin banyak perbedaan SNP, makin jauh hubungan kekerabatan.

| Eksperimen | Fitur | Pertanyaan biologis |
|---|---|---|
| **E1** | SNP distance + clustering | Apakah isolat berkerabat dekat cenderung satu kelompok? |
| **E2** | SNP matrix → RF | Apakah posisi SNP tertentu membawa sinyal sumber isolasi? |
| **E3** | DNABERT embedding → RF | Apakah pola konteks sekuens mencerminkan asal-usul isolat? |
| **E4** | SNP + DNABERT → RF | Apakah dua representasi saling melengkapi? |

In [ ]:
import math, textwrap

# ambil variabel yang sudah dihitung di sel-sel sebelumnya
_w_m   = w_m   if 'w_m'   in dir() else float('nan')
_b_m   = b_m   if 'b_m'   in dir() else float('nan')
_sil1  = sil_e1 if 'sil_e1' in dir() else float('nan')
_ari1  = ari_e1 if 'ari_e1' in dir() else float('nan')
_sep   = _b_m / max(_w_m, 1)

_am    = all_metrics
_e2    = _am.get('snp_only',    {})
_e3    = _am.get('dnabert_only',{})
_e4    = _am.get('hybrid',      {})
_e5    = _am.get('kmer_only',   {})

def _f(d, k): return d.get(k, float('nan'))

_e2f1  = _f(_e2, 'f1_macro');  _e2ba = _f(_e2, 'balanced_accuracy')
_e3f1  = _f(_e3, 'f1_macro');  _e3ba = _f(_e3, 'balanced_accuracy')
_e4f1  = _f(_e4, 'f1_macro');  _e4ba = _f(_e4, 'balanced_accuracy')
_e5f1  = _f(_e5, 'f1_macro')

_n_iso = len(metadata_df)
_n_snp = snp_df.shape[1] if 'snp_df' in dir() else 0
_n_cl  = metadata_df['snp_cluster'].nunique() if 'snp_cluster' in metadata_df.columns else 0
_sev   = metadata_df.get('serovar', pd.Series(dtype=str)).value_counts()
_top_sv= _sev.index[0] if len(_sev) else 'N/A'

def _wrap(text, width=88, indent='    '):
    return '\n'.join(textwrap.fill(p, width, initial_indent=indent,
                                   subsequent_indent=indent)
                     for p in text.split('\n'))

SEP = '=' * 88
sec = lambda t: f"\n{SEP}\n  {t}\n{SEP}"

lines = []

# E1 — SNP distance & clustering
lines.append(sec("E1 — SNP Distance & Hierarchical Clustering"))

_sil_desc = (
    "sangat baik (cluster genomik terpisah dengan jelas)" if _sil1 > 0.5 else
    "cukup baik (ada tumpang tindih parsial antar cluster)" if _sil1 > 0.2 else
    "rendah (cluster genomik tumpang tindih, batas kekerabatan tidak tegas)"
) if not math.isnan(_sil1) else "tidak dapat dihitung"

_ari_desc = (
    "sangat tinggi — pengelompokan genomik konsisten dengan klasifikasi NCBI" if _ari1 > 0.5 else
    "rendah hingga parsial — korespondensi terbatas antara cluster genomik dan SNP cluster NCBI" if _ari1 > 0.1 else
    "rendah — pengelompokan genomik dari Ward-linkage berbeda dari SNP cluster NCBI"
) if not math.isnan(_ari1) else "tidak dapat dihitung"

lines.append(_wrap(
    f"Dataset terdiri dari {_n_iso} isolat Salmonella enterica serovar {_top_sv} "
    f"dengan {_n_snp:,} posisi SNP yang terdeteksi relatif terhadap referensi "
    f"{cfg['data']['reference_genome']}. SNP adalah perbedaan nukleotida tunggal "
    f"yang timbul dari mutasi (substitusi, insersi, delesi) selama divergensi evolusioner."
))
lines.append("")

if not math.isnan(_w_m) and not math.isnan(_b_m):
    lines.append(_wrap(
        f"Rata-rata jarak SNP dalam satu cluster hierarchical (Ward): {_w_m:.1f} SNP. "
        f"Rata-rata jarak SNP antar cluster hierarchical berbeda: {_b_m:.1f} SNP. "
        f"Rasio separasi (hierarchical Ward, k={hclust_labels.nunique()}) = {_sep:.1f}×."
    ))
    lines.append("")
    if _sep >= 5:
        _sep_interp = (
            f"Rasio separasi {_sep:.1f}× menunjukkan bahwa isolat dalam cluster yang sama "
            f"jauh lebih berkerabat satu sama lain dibanding isolat dari cluster berbeda. "
            f"Secara biologis, ini mengindikasikan bahwa clustering Ward-linkage berhasil "
            f"memisahkan kelompok isolat dengan leluhur genomik yang berbeda. "
            f"Dalam investigasi forensik pangan, kelompok isolat dengan jarak SNP kecil "
            f"(≲{int(_w_m * 2)} SNP) merupakan kandidat rantai kontaminasi dari sumber yang sama."
        )
    elif _sep >= 2:
        _sep_interp = (
            f"Rasio separasi {_sep:.1f}× menunjukkan adanya pemisahan genomik antar cluster, "
            f"namun batas kekerabatan belum tegas. Beberapa isolat dari cluster berbeda masih "
            f"memiliki jarak SNP yang berdekatan, yang dapat berarti terdapat hubungan "
            f"kekerabatan parsial atau ambiguitas pada batas lineage."
        )
    else:
        _sep_interp = (
            f"Rasio separasi rendah ({_sep:.1f}×) menunjukkan bahwa isolat dari cluster berbeda "
            f"pun memiliki jarak SNP yang mirip. Ini dapat disebabkan oleh dataset yang relatif "
            f"homogen secara genomik, jumlah cluster yang terlalu besar, atau variasi SNP "
            f"yang kurang informatif untuk memisahkan lineage."
        )
    lines.append(_wrap(_sep_interp))
    lines.append("")

if not math.isnan(_sil1):
    lines.append(_wrap(
        f"Silhouette Score E1 = {_sil1:.4f} ({_sil_desc}). "
        f"Nilai ini mengukur seberapa rapat isolat di dalam clusternya dibanding cluster terdekat, "
        f"berdasarkan matriks jarak SNP pairwise."
    ))
    lines.append("")

if not math.isnan(_ari1):
    lines.append(_wrap(
        f"ARI (Adjusted Rand Index) antara hasil hierarchical clustering dan label SNP cluster NCBI "
        f"= {_ari1:.4f} ({_ari_desc}). "
        f"NCBI Pathogen Detection menetapkan SNP cluster berdasarkan pipeline genomik independen; "
        f"ARI yang tinggi berarti clustering Ward-linkage dari matriks SNP lokal menghasilkan "
        f"pengelompokan yang konsisten dengan referensi tersebut."
    ))

# ─────────────────────────────────────────────────────────────────────────────
# E2 — SNP-only Random Forest
# ─────────────────────────────────────────────────────────────────────────────
lines.append(sec("E2 — SNP-only Random Forest (ML Baseline)"))

if not math.isnan(_e2f1):
    if _e2f1 >= 0.70:
        _e2_bio = (
            f"Macro F1 = {_e2f1:.4f} (Balanced Accuracy = {_e2ba:.4f}). "
            f"Performa ini menunjukkan bahwa posisi-posisi SNP yang terdeteksi membawa sinyal "
            f"genomik yang berkorelasi dengan sumber isolasi. Secara biologis, ini konsisten "
            f"dengan hipotesis bahwa isolat dari sumber yang sama cenderung memiliki pola "
            f"mutasi yang lebih mirip — kemungkinan akibat paparan selektif terhadap lingkungan "
            f"atau inang yang serupa selama transmisi."
        )
    elif _e2f1 >= 0.40:
        _e2_bio = (
            f"Macro F1 = {_e2f1:.4f} (Balanced Accuracy = {_e2ba:.4f}). "
            f"Performa moderat ini mengindikasikan bahwa fitur SNP memiliki sebagian sinyal "
            f"genomik yang berkaitan dengan sumber isolasi, namun belum cukup untuk klasifikasi "
            f"yang konsisten. Hal ini dapat disebabkan oleh ketidakseimbangan kelas, jumlah "
            f"isolat yang terbatas per sumber, atau adanya sumber-sumber yang secara genomik "
            f"tumpang tindih (mis. food dan food_animal yang berbagi rantai pasokan hewan)."
        )
    else:
        _e2_bio = (
            f"Macro F1 = {_e2f1:.4f} (Balanced Accuracy = {_e2ba:.4f}). "
            f"Performa rendah menunjukkan bahwa fitur SNP dari dataset ini belum cukup "
            f"membedakan sumber isolasi secara konsisten. Kemungkinan penyebab: kelas yang "
            f"sangat tidak seimbang, SNP yang terlalu tersebar sehingga tidak spesifik per "
            f"sumber, atau label metadata yang terlalu heterogen untuk ditangkap oleh "
            f"model berbasis pola mutasi saja."
        )
    lines.append(_wrap(_e2_bio))
    lines.append("")
    lines.append(_wrap(
        "Catatan penting: hasil prediksi E2 tidak dapat diinterpretasikan sebagai bukti "
        "epidemiologis bahwa SNP tertentu adalah 'penanda sumber'. Model Random Forest "
        "berbasis pohon keputusan — ia menemukan pola diskriminatif dalam data latih, "
        "yang mungkin tidak merepresentasikan kausalitas biologis yang sebenarnya."
    ))
else:
    lines.append(_wrap("E2 (SNP-only) tidak berhasil dijalankan pada dataset ini."))

# ─────────────────────────────────────────────────────────────────────────────
# E3 — DNABERT embedding
# ─────────────────────────────────────────────────────────────────────────────
lines.append(sec("E3 — DNABERT-2 Embedding (Deep Learning Representation)"))

if not math.isnan(_e3f1):
    _e3_vs_e2 = (
        f"lebih tinggi dari E2 (SNP-only, F1={_e2f1:.4f}), "
        f"mengindikasikan bahwa embedding sekuens menangkap pola konteks nukleotida "
        f"yang tidak tertangkap oleh SNP eksplisit"
    ) if not math.isnan(_e2f1) and _e3f1 > _e2f1 else (
        f"lebih rendah dari E2 (SNP-only, F1={_e2f1:.4f}), "
        f"mengindikasikan bahwa pada dataset berukuran kecil ini, fitur mutasi eksplisit "
        f"lebih stabil dibanding embedding sekuens umum tanpa fine-tuning"
    ) if not math.isnan(_e2f1) else "sebanding dengan E2"

    lines.append(_wrap(
        f"Macro F1 = {_e3f1:.4f} (Balanced Accuracy = {_e3ba:.4f}). "
        f"Performa ini {_e3_vs_e2}. "
        f"DNABERT-2 digunakan sebagai feature extractor saja (tanpa fine-tuning) — "
        f"model memproses jendela sekuens DNA di sekitar posisi SNP dan menghasilkan "
        f"vektor 768 dimensi yang merepresentasikan konteks nukleotida."
    ))
    lines.append("")
    lines.append(_wrap(
        "Secara biologis, embedding yang berhasil berarti DNABERT menangkap pola "
        "komposisi dan konteks sekuens yang secara implisit berkorelasi dengan asal "
        "isolat. Namun tanpa fine-tuning pada data Salmonella spesifik, representasi "
        "ini bersifat general dan mungkin tidak sensitif terhadap variasi genomik "
        "yang relevan untuk source attribution bakteri patogen."
    ))
else:
    lines.append(_wrap("E3 (DNABERT-only) tidak berhasil dijalankan pada dataset ini."))

# ─────────────────────────────────────────────────────────────────────────────
# E4 — Hybrid
# ─────────────────────────────────────────────────────────────────────────────
lines.append(sec("E4 — Hybrid SNP + DNABERT"))

if not math.isnan(_e4f1):
    _base_max = max(v for v in [_e2f1, _e3f1] if not math.isnan(v)) if \
                any(not math.isnan(v) for v in [_e2f1, _e3f1]) else float('nan')

    if not math.isnan(_base_max) and _e4f1 > _base_max + 0.02:
        _e4_bio = (
            f"Macro F1 = {_e4f1:.4f} — melampaui E2 dan E3 secara individual. "
            f"Hasil ini mendukung hipotesis bahwa fitur SNP eksplisit (menangkap perbedaan "
            f"nukleotida pada posisi tertentu) dan embedding DNABERT (menangkap konteks sekuens "
            f"di sekitar variasi tersebut) saling melengkapi. Representasi gabungan memberikan "
            f"profil genomik yang lebih kaya, yang secara biologis konsisten dengan gagasan "
            f"bahwa identitas isolat tidak hanya ditentukan oleh SNP individual tetapi juga "
            f"oleh pola sekuens yang mengapit variasi tersebut."
        )
    elif not math.isnan(_base_max) and _e4f1 >= _base_max - 0.02:
        _e4_bio = (
            f"Macro F1 = {_e4f1:.4f} — sebanding dengan mode terbaik individual. "
            f"Fitur SNP sudah cukup dominan sehingga penambahan embedding DNABERT tidak "
            f"memberikan peningkatan signifikan. Pada dataset kecil, penggabungan dua "
            f"representasi berisiko menambah noise dimensi tanpa tambahan sinyal biologis "
            f"yang bermakna, terutama karena DNABERT tidak di-fine-tune."
        )
    else:
        _e4_bio = (
            f"Macro F1 = {_e4f1:.4f} — di bawah mode terbaik individual. "
            f"Penggabungan fitur SNP dan DNABERT menghasilkan ruang fitur berdimensi tinggi "
            f"yang mungkin menimbulkan efek 'curse of dimensionality' pada dataset berukuran "
            f"kecil. Secara biologis, informasi dari DNABERT mungkin redundan atau bertentangan "
            f"dengan sinyal SNP eksplisit pada konteks dataset ini."
        )
    lines.append(_wrap(_e4_bio))
else:
    lines.append(_wrap("E4 (Hybrid) tidak berhasil dijalankan pada dataset ini."))

# ─────────────────────────────────────────────────────────────────────────────
# E5 — K-mer (jika aktif)
if not math.isnan(_e5f1) and _e5f1 > 0:
    lines.append(sec("E5 — K-mer Frequency (Alignment-free Baseline)"))
    _base_max5 = max(v for v in [_e2f1, _e3f1] if not math.isnan(v)) if \
                 any(not math.isnan(v) for v in [_e2f1, _e3f1]) else float('nan')
    _e5_comp = (
        f"kompetitif dibanding fitur berbasis alignment (E2/E3)" if not math.isnan(_base_max5) and _e5f1 >= _base_max5 - 0.05
        else f"di bawah fitur berbasis alignment (E2 F1={_e2f1:.4f}, E3 F1={_e3f1:.4f})"
    )
    lines.append(_wrap(
        f"Macro F1 = {_e5f1:.4f} — {_e5_comp}. "
        f"K-mer frequency menghitung proporsi setiap kombinasi {cfg.get('kmer',{}).get('k',4)}-nukleotida "
        f"tanpa memerlukan alignment. Secara biologis, komposisi k-mer mencerminkan preferensi "
        f"kodon, bias dinukleotida, dan komposisi GC dari genom. Jika k-mer kompetitif dengan "
        f"SNP alignment, maka komposisi nukleotida global sudah cukup mencirikan asal isolat. "
        f"Jika tidak, sinyal sumber tersebar pada posisi SNP spesifik yang hanya terdeteksi "
        f"melalui alignment."
    ))

# Confusion matrix — error biologis
lines.append(sec("Kesalahan Prediksi — Makna Biologis"))

lines.append(_wrap(
    "Kesalahan klasifikasi pada source attribution tidak selalu berarti model gagal — "
    "beberapa kesalahan justru konsisten secara biologis:"
))
lines.append("")
_error_table = [
    ("food ↔ food_animal / animal",
     "Makanan hewani (daging, unggas, susu, telur) adalah produk dari reservoir hewan. "
     "Isolat Salmonella yang mengkontaminasi pangan hewani bisa memiliki profil genomik "
     "yang sangat mirip dengan isolat dari hewan itu sendiri."),
    ("human ↔ food",
     "Infeksi Salmonella pada manusia sebagian besar bersumber dari kontaminasi pangan. "
     "Isolat klinis manusia yang genomiknya mirip dengan isolat pangan dapat "
     "mengindikasikan rantai penularan dari sumber pangan tersebut."),
    ("environment ↔ lainnya",
     "Isolat dari lingkungan (air, tanah, fasilitas pengolahan) dapat memiliki profil "
     "genomik yang beragam karena sumber lingkungan merupakan reservoir sekunder dengan "
     "paparan dari berbagai inang dan sumber."),
]
for src_pair, bio_reason in _error_table:
    lines.append(f"    [{src_pair}]")
    lines.append(_wrap(bio_reason))
    lines.append("")

# Batasan interpretasi
lines.append(sec("Batasan Interpretasi"))

lines.append(_wrap(
    "Penelitian ini tidak menggunakan isolat dari kasus kontaminasi pangan spesifik "
    "secara langsung. Oleh karena itu, hasil yang diperoleh tidak dapat digunakan untuk "
    "menyimpulkan sumber kontaminasi kasus nyata. Analisis ini merupakan simulasi forensik "
    "genomik menggunakan data publik Salmonella enterica dari NCBI Pathogen Detection "
    "untuk mendemonstrasikan bagaimana whole-genome SNP alignment, hierarchical clustering, "
    "dan embedding sekuens dapat membantu memahami kedekatan genomik antar-isolat."
))
lines.append("")
lines.append(_wrap(
    "Keterbatasan tambahan: (1) label isolat berasal dari metadata pelapor yang mungkin "
    "tidak konsisten; (2) DNABERT-2 tidak di-fine-tune pada data Salmonella sehingga "
    "representasinya bersifat general; (3) dataset berukuran kecil dapat menyebabkan "
    "ketidakstabilan estimasi metrik; (4) model tidak divalidasi pada dataset independen "
    "dari luar NCBI Pathogen Detection."
))

print('\n'.join(lines))

---
## Batasan & Risiko

> **Kalimat utama:** Penelitian ini merupakan **simulasi forensik genomik berbasis data publik**, bukan investigasi epidemiologis resmi terhadap kasus MBG.

Pipeline ini boleh menjawab: *"Bagaimana SNP alignment + DNABERT membantu source attribution Salmonella?"*  
Pipeline ini **tidak boleh mengklaim**: *"Model ini membuktikan sumber keracunan MBG berasal dari X."*

### Batasan Utama

| # | Batasan | Dampak | Mitigasi yang dilakukan |
|:--:|---|---|---|
| 1 | Dataset bukan isolat MBG asli | Tidak bisa simpulkan sumber MBG langsung | MBG = konteks aktual; dataset publik = bahan simulasi |
| 2 | Metadata `isolation_source` bisa noisy | Label bias/salah | Normalisasi label + hapus entri ambigu (`unknown`) |
| 3 | Isolat sangat mirip bisa bocor train-test | Performa tampak terlalu tinggi | Group-aware split berdasarkan `snp_cluster` |
| 4 | Dataset kecil untuk DNABERT | Embedding bisa tidak representatif | DNABERT sebagai feature extractor, bukan fine-tuning |
| 5 | Source group biologis overlap | `food`/`animal`/`environment` tidak terpisah jelas | Jelaskan keterkaitan rantai pangan sebagai alasan biologis |
| 6 | SNP distance bukan bukti epidemiologi penuh | Genom dekat ≠ pasti satu sumber | Interpretasi harus disertai metadata dan konteks lapangan |
| 7 | Bias database publik | NCBI tidak merepresentasikan semua wilayah | Batasi klaim pada dataset yang dianalisis saja |
| 8 | Reference genome memengaruhi SNP | Reference berbeda → SNP matrix berbeda | Satu reference tetap (`GCF_000006945.2`), didokumentasikan |
| 9 | Assembly quality bervariasi | Contig banyak/N-content tinggi ganggu analisis | QC genome: panjang, jumlah contig, fraksi N |


In [ ]:
import math, textwrap
from pathlib import Path

# RISK ASSESSMENT OTOMATIS (berdasarkan angka aktual pipeline)

WARNINGS  = []   # risiko yang terdeteksi (butuh perhatian)
NOTES     = []   # catatan informatif (tidak kritis)
MITIGATED = []   # risiko yang sudah dimitigasi

def _warn(msg):  WARNINGS.append(msg)
def _note(msg):  NOTES.append(msg)
def _ok(msg):    MITIGATED.append(msg)

SEP = '─' * 80

# 1. Disclaimer dataset 
_note(
    "Dataset bukan isolat MBG. Penelitian ini menggunakan data publik NCBI Pathogen "
    "Detection untuk mensimulasikan forensik genomik. MBG diposisikan sebagai konteks "
    "aktual keamanan pangan, bukan target investigasi langsung."
)

# 2. Class imbalance
src_counts = metadata_df['isolation_source'].value_counts()
tiny_classes = src_counts[src_counts < 3]
if len(tiny_classes):
    _warn(
        f"CLASS IMBALANCE: {len(tiny_classes)} kelas dengan < 3 sampel: "
        f"{tiny_classes.to_dict()}. Estimasi F1 per kelas tidak stabil untuk kelas kecil."
    )
else:
    _ok(f"Class balance: semua kelas memiliki ≥ 3 sampel. "
        f"Kelas terbesar: '{src_counts.index[0]}' ({src_counts.iloc[0]} isolat).")

# 2b. Distribusi target binary (ML label)
if TARGET_COL in metadata_df.columns:
    bin_counts = metadata_df[TARGET_COL].value_counts()
    imbalance_ratio = bin_counts.max() / max(bin_counts.min(), 1)
    if imbalance_ratio > 2.5:
        _warn(
            f'Target "{TARGET_COL}" tidak seimbang: {imbalance_ratio:.1f}x '
            f'({bin_counts.to_dict()}). class_weight="balanced" sudah aktif.'
        )
    else:
        _ok(f'Target "{TARGET_COL}" cukup seimbang: {imbalance_ratio:.1f}x '
            f'({bin_counts.to_dict()}).')

# 3. Data leakage assessment
if 'naive_metrics' in dir() and naive_metrics:
    max_delta = float('-inf')
    leakage_modes = []
    for mode in ['snp_only', 'dnabert_only', 'hybrid', 'kmer_only']:
        ga = all_metrics.get(mode, {}).get('f1_macro', float('nan'))
        nv = naive_metrics.get(mode, {}).get('f1_macro', float('nan'))
        if not math.isnan(ga) and not math.isnan(nv):
            delta = nv - ga
            max_delta = max(max_delta, delta)
            if delta > 0.15:
                leakage_modes.append(f"{mode} ΔF1={delta:+.3f}")
    if leakage_modes:
        _warn(
            f"LEAKAGE RISK TINGGI: {', '.join(leakage_modes)}. "
            f"Naive F1 jauh lebih tinggi dari group-aware F1 — menunjukkan bahwa split "
            f"tanpa kontrol SNP cluster menghasilkan overestimasi performa nyata."
        )
    elif max_delta != float('-inf') and max_delta > 0.05:
        _warn(
            f"LEAKAGE RISK MODERAT: ΔF1 maksimum = {max_delta:+.3f}. "
            f"Sebagian isolat berkerabat mungkin tersebar ke set test. "
            f"Hasil group-aware tetap digunakan sebagai laporan utama."
        )
    else:
        _ok(
            f"Leakage risk rendah: ΔF1 maks = {max_delta:+.3f} — "
            f"group-aware split efektif mengisolasi SNP cluster dari set test."
        )
else:
    _note(
        "Leakage assessment belum dijalankan. "
        "Jalankan sel 'Leakage Assessment' di atas untuk melihat ΔF1 naive vs group-aware."
    )

# 4. Dataset size
n_iso = len(metadata_df)
if n_iso < 30:
    _warn(
        f"DATASET KECIL: {n_iso} isolat. Performa model bervariasi tinggi pada dataset "
        f"kecil; estimasi Macro F1 tidak stabil. Disarankan ≥ 50 isolat per kelas."
    )
elif n_iso < 60:
    _note(
        f"Dataset moderat: {n_iso} isolat. Cukup untuk demonstrasi pipeline, tetapi "
        f"generalisasi ke populasi lebih luas perlu dataset yang lebih besar."
    )
else:
    _ok(f"Dataset: {n_iso} isolat — ukuran memadai untuk eksperimen ini.")

# 5. Genomic separation (homogeneity)
_w_m_l = w_m if 'w_m' in dir() else float('nan')
_b_m_l = b_m if 'b_m' in dir() else float('nan')
_sep_l = _b_m_l / max(_w_m_l, 1) if not math.isnan(_w_m_l) and not math.isnan(_b_m_l) else float('nan')
if not math.isnan(_sep_l):
    if _sep_l < 2:
        _warn(
            f"SNP KURANG DISKRIMINATIF: separation ratio = {_sep_l:.1f}×. "
            f"Jarak SNP within-cluster dan between-cluster hampir sama, sehingga "
            f"matriks SNP belum mampu memisahkan kelompok sumber dengan baik. "
            f"Ini bukan semata homogenitas dataset, melainkan sinyal SNP yang "
            f"tidak cukup informatif untuk membedakan sumber isolasi."
        )
    elif _sep_l < 4:
        _note(
            f"Separasi genomik moderat: {_sep_l:.1f}×. "
            f"Ada pemisahan antar cluster, namun batas lineage belum tegas."
        )
    else:
        _ok(f"Separasi genomik baik: {_sep_l:.1f}× — cluster terbentuk dengan jelas.")

# 6. DNABERT tanpa fine-tuning
_note(
    f"DNABERT-2 digunakan sebagai feature extractor tanpa fine-tuning "
    f"({cfg['dnabert']['model_id']}). Embedding bersifat general dan mungkin tidak "
    f"optimal untuk Salmonella. Fine-tuning pada data Salmonella spesifik bisa "
    f"meningkatkan performa embedding secara signifikan."
)

# 7. Reference genome
ref = cfg['data']['reference_genome']
_ok(
    f"Reference genome tetap: {ref} (Salmonella Typhimurium LT2). "
    f"Reference yang berbeda akan menghasilkan posisi SNP yang berbeda. "
    f"Pilihan ini didokumentasikan di config.yaml dan manifest.json."
)

# 8. Genome QC
if 'qc_df' in dir():
    n_fail = int((~qc_df['qc_pass']).sum()) if 'qc_pass' in qc_df.columns else 0
    if n_fail > 0:
        _note(
            f"QC: {n_fail} isolat dibuang karena gagal threshold kualitas "
            f"(panjang genome, jumlah contig, N-content). "
            f"Jumlah isolat yang dianalisis: {n_iso}."
        )
    else:
        _ok(f"QC: semua {n_iso} isolat lulus threshold kualitas genome.")

# 9. Metadata noise
n_sources_raw = metadata_df['isolation_source'].nunique()
_note(
    f"Label metadata dinormalisasi: {n_sources_raw} kategori unik. "
    f"Jika entri 'Not Provided' masih muncul, ini keterbatasan dataset (metadata mungkin"
    f" di-cache sebelum filter ambigu diperbarui). Set force_recompute: true dan hapus"
    f" artifacts/data/interim/ untuk membersihkan ulang."
)

# 10. NCBI bias
top_geo = metadata_df['geo_loc_name'].str.split(':').str[0].str.strip().value_counts()
dominant = top_geo.index[0] if len(top_geo) else 'N/A'
dominant_pct = top_geo.iloc[0] / len(metadata_df) * 100 if len(top_geo) else 0
if dominant_pct > 60:
    _warn(
        f"BIAS GEOGRAFIS: {dominant_pct:.0f}% isolat berasal dari '{dominant}'. "
        f"Hasil mungkin tidak generalizable ke wilayah lain. Klaim dibatasi pada "
        f"dataset yang dianalisis."
    )
else:
    _note(
        f"Distribusi geografis: {len(top_geo)} negara/wilayah. "
        f"Lokasi dominan: '{dominant}' ({dominant_pct:.0f}%). "
        f"Interpretasi tetap dibatasi pada dataset publik yang dianalisis."
    )

# PRINT RISK REPORT

def _pp(items, label, char='!'):
    if not items: return
    print(f'\n[{char * len(label)}] {label}')
    print(SEP)
    for i, msg in enumerate(items, 1):
        print(textwrap.fill(f'{i}. {msg}', 88,
                            initial_indent='  ', subsequent_indent='     '))

print('=' * 80)
print('  RISK ASSESSMENT — SalmoTrace-BERT')
print(f'  Dataset: {n_iso} isolat · Target: {TARGET_COL} · Ref: {ref}')
print('=' * 80)

_pp(WARNINGS,  'PERINGATAN AKTIF', '!')
_pp(NOTES,     'CATATAN PENTING',  'i')
_pp(MITIGATED, 'RISIKO DIMITIGASI','v')

print(f'\n{SEP}')
print(f'  Ringkasan: {len(WARNINGS)} peringatan, {len(NOTES)} catatan, '
      f'{len(MITIGATED)} risiko dimitigasi')
print(SEP)

# ── Simpan ke artifact ───
_lim_lines = ['=' * 80, '  RISK ASSESSMENT — SalmoTrace-BERT', '=' * 80, '']

def _collect(items, label):
    if not items: return
    _lim_lines.append(f'[{label}]')
    _lim_lines.append('─' * 80)
    for i, msg in enumerate(items, 1):
        _lim_lines.append(textwrap.fill(f'{i}. {msg}', 88,
                                        initial_indent='  ', subsequent_indent='     '))
    _lim_lines.append('')

_collect(WARNINGS,  'PERINGATAN AKTIF')
_collect(NOTES,     'CATATAN PENTING')
_collect(MITIGATED, 'RISIKO DIMITIGASI')
_lim_lines += [
    '─' * 80,
    '  DISCLAIMER:',
    textwrap.fill(
        'Penelitian ini merupakan simulasi forensik genomik berbasis data publik '
        'NCBI Pathogen Detection, bukan investigasi epidemiologis resmi. '
        'Hasil prediksi source attribution tidak dapat digunakan sebagai bukti '
        'sumber kontaminasi kasus nyata tanpa validasi epidemiologi independen.',
        88, initial_indent='  ', subsequent_indent='  ',
    ),
    '─' * 80,
]

lim_path = Path(_p('artifacts/reports/limitations_report.txt'))
lim_path.parent.mkdir(parents=True, exist_ok=True)
lim_path.write_text('\n'.join(_lim_lines), encoding='utf-8')
print(f'\nLimitations report saved: {lim_path}')